# DeepBudget-Vis — Literature-Based Benchmark Models

**CNN-LSTM · TCN-LSTM · LSTM-mTrans-MLP**

Built and executed against the **actual dataset in this repository** (branch
`Suren-1`): `train.csv`, `val.csv`, `test.csv`, `val_with_context.csv`,
`test_with_context.csv`.

These are **literature-based benchmark models** (equivalently, *literature-based
SOTA comparison models*) for comparison against the proposed DeepBudget-Vis /
HRC-DRGNet model. They are **not** claimed to constitute a universal
state-of-the-art ranking — the literature survey does not establish one.

---

## ⚠️ Target-name discrepancy — read this before citing anything

The task specification asks for targets `NET_PATIENT_REVENUE` and
`TOT_OVERALL_EXP`.

**`TOT_OVERALL_EXP` does not exist in this dataset.** Verified by direct column
inspection (Section 3). The dataset's second financial target is
**`TOTAL_OPERATING_EXP`**, and there is no column containing the substring
`OVERALL` at all.

This notebook therefore uses **`TOTAL_OPERATING_EXP`** and reports it under that
real name. It is treated as the intended second target (an explicit, documented
alias mapping — not a silent substitution). If `TOT_OVERALL_EXP` is a genuinely
different quantity that exists in a newer preprocessing run, this notebook is
**not** using it, and the mapping in the configuration cell must be corrected.

---

## Fair-comparison contract

All three models share: identical chronological split, identical sequence
construction, identical input features, identical target definitions and
scaling, identical loss (`SmoothL1Loss`), identical optimizer policy (`AdamW`),
identical `ReduceLROnPlateau` and early-stopping policy, and the same seed.
**Only the architecture differs.**


## Source verification note

I verified that both cited works exist and confirmed their **high-level model
composition**. I could **not** retrieve their implementation-level details
(layer counts, channel widths, hidden sizes, learning rates) — the publisher
pages returned access errors (MDPI `HTTP 403`; the other source failed TLS
certificate verification).

**Verified:**

- Kabir et al., *"LSTM–Transformer-Based Robust Hybrid Deep Learning Model for
  Financial Time Series Forecasting"*, **Sci** 7(1), 7 (2025). The abstract
  states the work proposes **LSTM-mTrans-MLP**, integrating an LSTM network, a
  **modified Transformer** network, and a multilayered perceptron.
  [mdpi.com/2413-4155/7/1/7](https://www.mdpi.com/2413-4155/7/1/7)
- A closely-matching hybrid multivariate-forecasting work compares exactly
  **CNN-LSTM, CNN-BiLSTM, TCN-LSTM, TCN-BiLSTM**, reporting TCN-BiLSTM best
  overall on its own two datasets (Traffic Volume R² ≈ 0.976, Air Quality
  R² ≈ 0.94). [d-nb.info/1353813266/34](https://d-nb.info/1353813266/34)

**Consequence:** every numeric architecture choice below (channels, kernel
sizes, dilation schedule, hidden sizes, head counts, MLP widths, dropout), the
shared loss, the optimizer and the scheduler settings are **implementation
assumptions**, labelled as such per model. None is presented as a paper value.

Those R² figures belong to **traffic / air-quality** data. They are not
comparable to, and say nothing about, results on hospital financial data.

*Source content paraphrased for licensing compliance.*


## 1. USER CONFIGURATION — the only cell you should need to edit

In [1]:
# ============================================================
# USER CONFIGURATION - EDIT THIS SECTION ONLY
# ============================================================
from pathlib import Path

# Repo root containing train.csv / val.csv / test.csv
BASE_DIR   = "."
DATA_DIR   = BASE_DIR
OUTPUT_DIR = f"{BASE_DIR}/benchmarks_repo_data/baseline_outputs_v1"

# Actual target columns in THIS dataset, in fixed output order.
#   output[:, 0] -> TARGET_COLUMNS[0]
#   output[:, 1] -> TARGET_COLUMNS[1]
TARGET_COLUMNS = [
    "NET_PATIENT_REVENUE",
    "TOTAL_OPERATING_EXP",   # requested as TOT_OVERALL_EXP, which does not exist here
]

# Documented alias mapping, printed in Section 3 for transparency.
REQUESTED_TARGET_ALIAS = {
    "NET_PATIENT_REVENUE": "NET_PATIENT_REVENUE",
    "TOT_OVERALL_EXP":     "TOTAL_OPERATING_EXP",
}

SEQUENCE_LENGTH = 56
RANDOM_SEED     = 42
BATCH_SIZE      = 128
EPOCHS          = 100
PATIENCE        = 15
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-4

RESUME = True          # continue from latest checkpoint if one exists

# Numerical-stability / performance switches (implementation configuration)
USE_AMP        = False   # kept False: fp16 overflow is a known NaN source here
GRAD_CLIP_NORM = 1.0     # None to disable
NUM_WORKERS    = 0

# Scheduler (implementation configuration, NOT paper-specified)
SCHED_FACTOR   = 0.5
SCHED_PATIENCE = 5
SCHED_MIN_LR   = 1e-7

# Metric options
MAPE_EPSILON            = 1.0    # denominator floor, in original target units
TRAIN_METRICS_EVAL_PASS = False  # True = extra no-dropout pass for train metrics

MODELS_TO_RUN = ["cnn_lstm", "tcn_lstm", "lstm_mtrans_mlp"]
# ============================================================
# END OF USER CONFIGURATION
# ============================================================
print("Config loaded.")
print("  DATA_DIR      :", DATA_DIR)
print("  OUTPUT_DIR    :", OUTPUT_DIR)
print("  TARGET_COLUMNS:", TARGET_COLUMNS)
print("  SEQUENCE_LENGTH:", SEQUENCE_LENGTH)


Config loaded.
  DATA_DIR      : .
  OUTPUT_DIR    : ./benchmarks_repo_data/baseline_outputs_v1
  TARGET_COLUMNS: ['NET_PATIENT_REVENUE', 'TOTAL_OPERATING_EXP']
  SEQUENCE_LENGTH: 56


## 2. Imports, reproducibility, device

In [2]:
import os, json, time, random, copy, warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils import weight_norm

from sklearn.metrics import (r2_score, explained_variance_score,
                             median_absolute_error, max_error)

import matplotlib
matplotlib.use("Agg")           # save figures to disk; keeps notebook size small
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 20, "axes.titlesize": 20, "axes.labelsize": 20,
    "xtick.labelsize": 20, "ytick.labelsize": 20, "legend.fontsize": 20,
    "figure.dpi": 300, "savefig.dpi": 300,
})


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")

print("Device:", "CUDA" if DEVICE.type == "cuda" else "CPU")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("AMP enabled:", AMP_ENABLED)
print("Seed:", RANDOM_SEED, "| torch:", torch.__version__)

DATA_DIR_P = Path(DATA_DIR)
OUT_P = Path(OUTPUT_DIR)


Device: CPU
AMP enabled: False
Seed: 42 | torch: 2.8.0+cu128


## 3. Load the supplied splits and verify them

The five CSVs are used exactly as supplied. Nothing is re-split, re-scaled or
re-imputed.

In [3]:
REQUIRED = ["train.csv", "val.csv", "test.csv"]
for fn in REQUIRED:
    if not (DATA_DIR_P / fn).exists():
        raise FileNotFoundError(
            f"Required file missing: {DATA_DIR_P / fn}\n"
            "Fix DATA_DIR in the configuration cell. No data is created or downloaded."
        )

train_df = pd.read_csv(DATA_DIR_P / "train.csv", parse_dates=["DATE"])
val_df   = pd.read_csv(DATA_DIR_P / "val.csv",   parse_dates=["DATE"])
test_df  = pd.read_csv(DATA_DIR_P / "test.csv",  parse_dates=["DATE"])

print("=" * 72)
print("SUPPLIED SPLITS")
print("=" * 72)
for nm, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    months = (d.DATE.max().to_period("M") - d.DATE.min().to_period("M")).n + 1
    print(f"  {nm:6s} rows={len(d):5d} cols={d.shape[1]:3d} "
          f"{d.DATE.min().date()} .. {d.DATE.max().date()}  "
          f"months={months}  dup_dates={d.DATE.duplicated().sum()}")

# ---- target-name resolution, explicit and printed ----
print("\n" + "=" * 72)
print("TARGET RESOLUTION")
print("=" * 72)
all_cols = list(train_df.columns)
overall_cols = [c for c in all_cols if "OVERALL" in c.upper()]
print("  Columns containing 'OVERALL':", overall_cols if overall_cols else "NONE")
for requested, actual in REQUESTED_TARGET_ALIAS.items():
    exists_req = requested in all_cols
    exists_act = actual in all_cols
    if not exists_act:
        raise KeyError(f"Configured target column '{actual}' not found in the dataset.")
    if requested == actual:
        print(f"  '{requested}' -> found directly.")
    else:
        print(f"  '{requested}' NOT FOUND -> mapped to '{actual}' "
              f"(exists={exists_act}). Documented alias, not a silent substitution.")

TARGETS = list(TARGET_COLUMNS)
N_TARGETS = len(TARGETS)
print("\n  TARGET INDEX MAPPING (enforced everywhere):")
for i, t in enumerate(TARGETS):
    print(f"     output[:, {i}] = {t}")


SUPPLIED SPLITS
  train  rows= 2557 cols= 85 2016-01-01 .. 2022-12-31  months=84  dup_dates=0
  val    rows=  547 cols= 85 2023-01-01 .. 2024-06-30  months=18  dup_dates=0
  test   rows=  549 cols= 85 2024-07-01 .. 2025-12-31  months=18  dup_dates=0

TARGET RESOLUTION
  Columns containing 'OVERALL': NONE
  'NET_PATIENT_REVENUE' -> found directly.
  'TOT_OVERALL_EXP' NOT FOUND -> mapped to 'TOTAL_OPERATING_EXP' (exists=True). Documented alias, not a silent substitution.

  TARGET INDEX MAPPING (enforced everywhere):
     output[:, 0] = NET_PATIENT_REVENUE
     output[:, 1] = TOTAL_OPERATING_EXP


In [4]:
# ---- feature columns: the supplied *_scaled features, targets excluded ----
EXCLUDE = {"DATE", "IS_ANOMALY", "ANOMALY_TYPE", *TARGETS}
RAW_FEATURES = [c for c in train_df.columns
                if not c.endswith("_scaled") and c not in EXCLUDE]
FEATURE_COLS = [f"{c}_scaled" for c in RAW_FEATURES]
TARGET_COLS_SCALED = [f"{t}_scaled" for t in TARGETS]

missing = [c for c in FEATURE_COLS + TARGET_COLS_SCALED if c not in train_df.columns]
if missing:
    raise KeyError(f"Missing expected scaled columns: {missing[:20]}")

print(f"Feature columns used: {len(FEATURE_COLS)} (pre-scaled, supplied by your preprocessing)")
print("IS_ANOMALY / ANOMALY_TYPE are EXCLUDED from the feature matrix "
      "(labels, not forecasting inputs).")
print("\nFirst 10 feature columns:", FEATURE_COLS[:10])


Feature columns used: 39 (pre-scaled, supplied by your preprocessing)
IS_ANOMALY / ANOMALY_TYPE are EXCLUDED from the feature matrix (labels, not forecasting inputs).

First 10 feature columns: ['YEAR_scaled', 'IS_WEEKEND_scaled', 'QUARTER_scaled', 'IS_HOLIDAY_scaled', 'OCCUPANCY_RATE_scaled', 'STAFFED_BEDS_scaled', 'ADMISSIONS_scaled', 'ER_VISITS_scaled', 'OP_VISITS_scaled', 'SURGERIES_scaled']


### 3.1 Leakage audit of the supplied scaling

Claim to verify: every `_scaled` column is a z-score fitted on **train only**,
and the targets use `log1p` then a train-only z-score. If true, the supplied
preprocessing carries no val/test leakage and can be used as-is.

In [5]:
print("=" * 72)
print("SCALING PROVENANCE CHECK")
print("=" * 72)

worst_feat = 0.0
for c in RAW_FEATURES:
    mu, sd = train_df[c].mean(), train_df[c].std(ddof=0)
    if sd == 0:
        continue
    for d in (val_df, test_df):
        err = float(np.max(np.abs((d[c] - mu) / sd - d[f"{c}_scaled"])))
        worst_feat = max(worst_feat, err)
print(f"  Features: max |recomputed(train mu,sd) - supplied _scaled| on val+test "
      f"= {worst_feat:.3e}")

TARGET_LOG_MU, TARGET_LOG_SD = {}, {}
worst_targ = 0.0
for t in TARGETS:
    lg = np.log1p(train_df[t])
    mu, sd = float(lg.mean()), float(lg.std(ddof=0))
    TARGET_LOG_MU[t], TARGET_LOG_SD[t] = mu, sd
    for d in (train_df, val_df, test_df):
        err = float(np.max(np.abs((np.log1p(d[t]) - mu) / sd - d[f"{t}_scaled"])))
        worst_targ = max(worst_targ, err)
    print(f"  {t}: log1p+zscore train-fit  mu={mu:.6f} sd={sd:.6f}")
print(f"  Targets: max reconstruction error across all splits = {worst_targ:.3e}")

if worst_feat > 1e-8 or worst_targ > 1e-6:
    raise AssertionError(
        "Supplied _scaled columns are NOT reproducible from train-only statistics. "
        "The scaling provenance cannot be trusted; stopping rather than proceeding."
    )
print("\n  VERDICT: all scaling is train-fit only. No preprocessing leakage. "
      "Supplied columns used as-is.")


SCALING PROVENANCE CHECK
  Features: max |recomputed(train mu,sd) - supplied _scaled| on val+test = 1.776e-15
  NET_PATIENT_REVENUE: log1p+zscore train-fit  mu=11.859657 sd=0.147880
  TOTAL_OPERATING_EXP: log1p+zscore train-fit  mu=11.515974 sd=0.077285
  Targets: max reconstruction error across all splits = 2.309e-14

  VERDICT: all scaling is train-fit only. No preprocessing leakage. Supplied columns used as-is.


## 4. Sequence construction on one continuous series

The three splits are **contiguous, gap-free daily** records (verified below), so
they are concatenated into a single chronological series. A window for a target
at time `t` is `[t-L, t-1]` — strictly historical.

**Why concatenate rather than window each split independently?** Windowing each
split alone would discard the first `L` days of validation and test (no history
available), and would waste the `*_with_context.csv` files' purpose. Your
supplied `val_with_context.csv` / `test_with_context.csv` already encode exactly
this idea: they prepend 30 rows copied verbatim from the *preceding* split so the
earliest val/test windows have history. Concatenation generalises that to any
`L`, so **every** validation and test target remains predictable at
`SEQUENCE_LENGTH = 56`.

**Why this is leakage-free.** A window for a validation target may include train
rows, and a window for a test target may include validation rows — these are
observations strictly *before* the prediction time, i.e. legitimately known at
inference. The target row `t` itself is never inside its own window, so
same-timestamp derived features (e.g. `OPERATING_MARGIN_PCT`, which is an exact
function of both targets at time `t`) cannot leak the answer. Split membership is
assigned by the **target date**, so no target ever changes partition.

In [6]:
full_df = (pd.concat([train_df, val_df, test_df], ignore_index=True)
             .sort_values("DATE").reset_index(drop=True))

n_expected = (full_df.DATE.max() - full_df.DATE.min()).days + 1
step_counts = full_df.DATE.diff().dropna().dt.days.value_counts().to_dict()

print("=" * 72)
print("CONTINUOUS SERIES CHECK")
print("=" * 72)
print(f"  rows={len(full_df)}  {full_df.DATE.min().date()} .. {full_df.DATE.max().date()}")
print(f"  duplicate dates : {full_df.DATE.duplicated().sum()}")
print(f"  day-step counts : {step_counts}   (must be exactly {{1: n-1}})")
print(f"  expected daily rows={n_expected}  actual={len(full_df)}")

if full_df.DATE.duplicated().any():
    raise AssertionError("Duplicate dates in the concatenated series.")
if set(step_counts.keys()) != {1}:
    raise AssertionError(f"Series is not gap-free daily; day steps found: {step_counts}")
if n_expected != len(full_df):
    raise AssertionError("Row count does not match the calendar span.")
print("  VERDICT: perfectly contiguous daily series. Concatenation is valid.")

TRAIN_END = train_df.DATE.max()
VAL_END   = val_df.DATE.max()

F_ALL = full_df[FEATURE_COLS].to_numpy(np.float32)
Y_ALL = full_df[TARGET_COLS_SCALED].to_numpy(np.float32)
DATES = full_df["DATE"].to_numpy()

L = int(SEQUENCE_LENGTH)
if L >= len(full_df):
    raise ValueError(f"SEQUENCE_LENGTH={L} too large for {len(full_df)} rows.")

idx_by_split = {"train": [], "val": [], "test": []}
for i in range(L, len(full_df)):
    d = full_df.DATE.iloc[i]
    if d <= TRAIN_END:
        idx_by_split["train"].append(i)
    elif d <= VAL_END:
        idx_by_split["val"].append(i)
    else:
        idx_by_split["test"].append(i)


def build(indices):
    X = np.stack([F_ALL[i - L:i] for i in indices]).astype(np.float32)
    y = np.stack([Y_ALL[i] for i in indices]).astype(np.float32)
    return X, y, np.array([DATES[i] for i in indices])


X_train, y_train, dt_train = build(idx_by_split["train"])
X_val,   y_val,   dt_val   = build(idx_by_split["val"])
X_test,  y_test,  dt_test  = build(idx_by_split["test"])

INPUT_DIM = X_train.shape[-1]

print("\n" + "=" * 72)
print("SEQUENCE SHAPES")
print("=" * 72)
print("Train X shape:          ", X_train.shape)
print("Train y shape:          ", y_train.shape)
print("Validation X shape:     ", X_val.shape)
print("Validation y shape:     ", y_val.shape)
print("Test X shape:           ", X_test.shape)
print("Test y shape:           ", y_test.shape)
print("Number of input features:", INPUT_DIM)
print("Sequence length:        ", L)
print()
print(f"Train targets: {dt_train.min()} .. {dt_train.max()}  (n={len(dt_train)})")
print(f"Val   targets: {dt_val.min()} .. {dt_val.max()}  (n={len(dt_val)})")
print(f"Test  targets: {dt_test.min()} .. {dt_test.max()}  (n={len(dt_test)})")

# every official val/test row must survive as a predictable target
assert len(dt_val) == len(val_df), (len(dt_val), len(val_df))
assert len(dt_test) == len(test_df), (len(dt_test), len(test_df))
print(f"\nAll {len(val_df)} validation and {len(test_df)} test targets retained "
      f"(none lost to the lookback window).")
print(f"Train targets = {len(train_df)} - {L} = {len(train_df) - L} "
      f"(first {L} days have no full history).")


CONTINUOUS SERIES CHECK
  rows=3653  2016-01-01 .. 2025-12-31
  duplicate dates : 0
  day-step counts : {1: 3652}   (must be exactly {1: n-1})
  expected daily rows=3653  actual=3653
  VERDICT: perfectly contiguous daily series. Concatenation is valid.

SEQUENCE SHAPES
Train X shape:           (2501, 56, 39)
Train y shape:           (2501, 2)
Validation X shape:      (547, 56, 39)
Validation y shape:      (547, 2)
Test X shape:            (549, 56, 39)
Test y shape:            (549, 2)
Number of input features: 39
Sequence length:         56

Train targets: 2016-02-26T00:00:00.000000000 .. 2022-12-31T00:00:00.000000000  (n=2501)
Val   targets: 2023-01-01T00:00:00.000000000 .. 2024-06-30T00:00:00.000000000  (n=547)
Test  targets: 2024-07-01T00:00:00.000000000 .. 2025-12-31T00:00:00.000000000  (n=549)

All 547 validation and 549 test targets retained (none lost to the lookback window).
Train targets = 2557 - 56 = 2501 (first 56 days have no full history).


## 5. Data validation gate

In [7]:
print("=" * 72)
print("DATA VALIDATION")
print("=" * 72)
problems = []

def chk(label, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f"  -- {detail}" if detail else ""))
    if not ok:
        problems.append(label)

chk("Dataset found", True, "train.csv / val.csv / test.csv")
chk("Train split found", len(X_train) > 0, f"{len(X_train)} sequences")
chk("Validation split found", len(X_val) > 0, f"{len(X_val)} sequences")
chk("Test split found", len(X_test) > 0, f"{len(X_test)} sequences")
for t in TARGETS:
    chk(f"Target {t} found", t in full_df.columns)
chk("Feature dimensions consistent",
    X_train.shape[-1] == X_val.shape[-1] == X_test.shape[-1], f"{INPUT_DIM}")
chk("Sequence dimensions valid",
    X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == L, f"L={L}")
chk("No unexpected target mismatch",
    y_train.shape[1] == y_val.shape[1] == y_test.shape[1] == N_TARGETS, f"{N_TARGETS}")
chk("Chronological order: train < val < test",
    dt_train.max() < dt_val.min() < dt_val.max() < dt_test.min())
chk("No target-date overlap between splits",
    len(set(dt_train) & set(dt_val)) == 0 and len(set(dt_val) & set(dt_test)) == 0)

fin = True
for nm, a in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test),
              ("y_train", y_train), ("y_val", y_val), ("y_test", y_test)]:
    nn_, ni = int(np.isnan(a).sum()), int(np.isinf(a).sum())
    if nn_ or ni:
        fin = False
        print(f"        !! {nm}: {nn_} NaN, {ni} Inf")
chk("No NaN/Inf in model input", fin)
chk("Input magnitudes bounded (pre-scaled)",
    float(np.abs(X_train).max()) < 100.0,
    f"max|X_train| = {float(np.abs(X_train).max()):.3f}")

if problems:
    raise ValueError("Validation failed: " + ", ".join(problems))
print("\nAll validation checks passed.")


DATA VALIDATION
[PASS] Dataset found  -- train.csv / val.csv / test.csv
[PASS] Train split found  -- 2501 sequences
[PASS] Validation split found  -- 547 sequences
[PASS] Test split found  -- 549 sequences
[PASS] Target NET_PATIENT_REVENUE found
[PASS] Target TOTAL_OPERATING_EXP found
[PASS] Feature dimensions consistent  -- 39
[PASS] Sequence dimensions valid  -- L=56
[PASS] No unexpected target mismatch  -- 2
[PASS] Chronological order: train < val < test
[PASS] No target-date overlap between splits
[PASS] No NaN/Inf in model input
[PASS] Input magnitudes bounded (pre-scaled)  -- max|X_train| = 15.965

All validation checks passed.


## 6. Output folder structure

In [8]:
MODEL_KEYS = ["cnn_lstm", "tcn_lstm", "lstm_mtrans_mlp"]
MODEL_DISPLAY = {"cnn_lstm": "CNN-LSTM", "tcn_lstm": "TCN-LSTM",
                 "lstm_mtrans_mlp": "LSTM-mTrans-MLP"}
SUBDIRS = ["checkpoints/latest", "checkpoints/best", "metrics",
           "predictions", "plots", "logs", "config"]

for mk in MODEL_KEYS:
    for sd in SUBDIRS:
        (OUT_P / mk / sd).mkdir(parents=True, exist_ok=True)
(OUT_P / "comparison" / "plots").mkdir(parents=True, exist_ok=True)

def mdir(mk, *parts):
    return OUT_P / mk / Path(*parts)

print("Output tree under:", OUT_P.resolve())
for mk in MODEL_KEYS:
    print("  " + mk + "/  " + " ".join(SUBDIRS))
print("  comparison/plots/")


Output tree under: /projects/sandbox/Advertisement_effect/benchmarks_repo_data/baseline_outputs_v1
  cnn_lstm/  checkpoints/latest checkpoints/best metrics predictions plots logs config
  tcn_lstm/  checkpoints/latest checkpoints/best metrics predictions plots logs config
  lstm_mtrans_mlp/  checkpoints/latest checkpoints/best metrics predictions plots logs config
  comparison/plots/


## 7. DataLoaders

Only the training loader is shuffled. Shuffling training *windows* does not
break temporal integrity — each window is a self-contained
`(history → next-day target)` example and the split boundaries are untouched.
Validation and test order is preserved so predictions stay chronological.

In [9]:
def make_loader(X, y, shuffle):
    return DataLoader(
        TensorDataset(torch.from_numpy(X), torch.from_numpy(y)),
        batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

train_loader = make_loader(X_train, y_train, True)
val_loader   = make_loader(X_val,   y_val,   False)
test_loader  = make_loader(X_test,  y_test,  False)

print(f"train batches={len(train_loader)} (shuffled)")
print(f"val   batches={len(val_loader)} (order preserved)")
print(f"test  batches={len(test_loader)} (order preserved)")
xb, yb = next(iter(train_loader))
print("sample batch X:", tuple(xb.shape), " y:", tuple(yb.shape))


train batches=20 (shuffled)
val   batches=5 (order preserved)
test  batches=5 (order preserved)
sample batch X: (128, 56, 39)  y: (128, 2)


## 8. Metrics — exactly the ten required, per target

`MAE, RMSE, MAPE, sMAPE, R2, ExplainedVar, MedianAE, MaxError, MAE_scaled, RMSE_scaled`

- The 8 primary metrics are computed in **original financial units** (dollars),
  by inverting the supplied `log1p` + train-z-score transform.
- `MAPE` / `sMAPE` use a floored denominator (`MAPE_EPSILON`). Both targets are
  well above zero here (min ≈ \$77k), so MAPE is well-behaved for this dataset.
- `MAE_scaled = MAE / train_target_std`, `RMSE_scaled = RMSE / train_target_std`,
  using the **train** std in original units for every split, so the values are
  comparable across splits and models.
- Reported **separately per target**, never averaged into one score.

In [10]:
METRIC_NAMES = ["MAE", "RMSE", "MAPE", "sMAPE", "R2",
                "ExplainedVar", "MedianAE", "MaxError",
                "MAE_scaled", "RMSE_scaled"]


def to_original_units(y_scaled):
    """Invert log1p + train z-score, per target."""
    y_scaled = np.asarray(y_scaled, dtype=np.float64)
    out = np.empty_like(y_scaled)
    for i, t in enumerate(TARGETS):
        out[:, i] = np.expm1(y_scaled[:, i] * TARGET_LOG_SD[t] + TARGET_LOG_MU[t])
    return out


y_train_orig = to_original_units(y_train)
TRAIN_TARGET_STD = y_train_orig.std(axis=0, ddof=0)
if np.any(TRAIN_TARGET_STD <= 0) or not np.all(np.isfinite(TRAIN_TARGET_STD)):
    raise ValueError(f"Invalid train target std: {TRAIN_TARGET_STD}")

print("Train-target std (original units), used for *_scaled metrics:")
for i, t in enumerate(TARGETS):
    print(f"   {t}: {TRAIN_TARGET_STD[i]:,.4f}")


def metrics_1d(y_true, y_pred, train_std):
    y_true = np.asarray(y_true, np.float64).ravel()
    y_pred = np.asarray(y_pred, np.float64).ravel()
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() != y_true.size:
        warnings.warn(f"Dropped {y_true.size - int(m.sum())} non-finite pair(s).")
    y_true, y_pred = y_true[m], y_pred[m]
    if y_true.size == 0:
        return {k: float("nan") for k in METRIC_NAMES}

    err = y_true - y_pred
    ae = np.abs(err)
    mae = float(ae.mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    mape = float((ae / np.maximum(np.abs(y_true), MAPE_EPSILON)).mean() * 100)
    smape = float((2 * ae / np.maximum(np.abs(y_true) + np.abs(y_pred),
                                       MAPE_EPSILON)).mean() * 100)
    if np.var(y_true) <= 0 or y_true.size < 2:
        r2 = evs = float("nan")
    else:
        r2 = float(r2_score(y_true, y_pred))
        evs = float(explained_variance_score(y_true, y_pred))
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "sMAPE": smape,
            "R2": r2, "ExplainedVar": evs,
            "MedianAE": float(median_absolute_error(y_true, y_pred)),
            "MaxError": float(max_error(y_true, y_pred)),
            "MAE_scaled": float(mae / train_std),
            "RMSE_scaled": float(rmse / train_std)}


def metrics_all(y_true_s, y_pred_s):
    yt, yp = to_original_units(y_true_s), to_original_units(y_pred_s)
    return {t: metrics_1d(yt[:, i], yp[:, i], TRAIN_TARGET_STD[i])
            for i, t in enumerate(TARGETS)}


def flatten(prefix, per_target):
    return {f"{prefix}_{t}_{k}": v
            for t, d in per_target.items() for k, v in d.items()}


# self-test of the maths on synthetic values
_a = np.array([100.0, 200.0, 300.0, 400.0])
_b = _a + np.array([10.0, -10.0, 10.0, -10.0])
_m = metrics_1d(_a, _b, 100.0)
assert abs(_m["MAE"] - 10) < 1e-9 and abs(_m["RMSE"] - 10) < 1e-9
assert abs(_m["MAE_scaled"] - 0.1) < 1e-9 and abs(_m["MaxError"] - 10) < 1e-9
print("\nMetric self-test passed (MAE / RMSE / MAE_scaled / MaxError).")
print("Metrics tracked:", METRIC_NAMES)


Train-target std (original units), used for *_scaled metrics:
   NET_PATIENT_REVENUE: 20,443.4222
   TOTAL_OPERATING_EXP: 7,604.5679

Metric self-test passed (MAE / RMSE / MAE_scaled / MaxError).
Metrics tracked: ['MAE', 'RMSE', 'MAPE', 'sMAPE', 'R2', 'ExplainedVar', 'MedianAE', 'MaxError', 'MAE_scaled', 'RMSE_scaled']


## 9. CNN-LSTM — Paper Fidelity

```
Paper:   "Enhanced Multivariate Time Series Forecasting"
Authors: A. Mahmoud and A. Mohammed
Venue:   Neural Processing Letters, 56(5), p.223
Model:   CNN-LSTM  (one of four compared: CNN-LSTM, CNN-BiLSTM,
                    TCN-LSTM, TCN-BiLSTM)
```

**Architecture described by the paper (high level):** multivariate time-series
input → convolutional feature extraction for local temporal patterns → LSTM for
sequential dependencies → forecasting output.

**Implementation here:**
`[B,T,F] → Conv1d → ReLU → Conv1d → ReLU → Dropout → LSTM → last step → Linear → [B,2]`

**Implementation assumptions — not explicitly specified in the paper** (and not
verifiable from the sources I could reach):

- 2 conv layers, 64 channels, kernel 3, `padding=1`
- LSTM: 2 layers, hidden 128, **unidirectional**
- Dropout 0.2; head `Linear(128 → 2)`
- Final LSTM time step used as the sequence representation

**Deliberately NOT added:** no bidirectional LSTM, no Transformer, no attention,
no dilation, no residual connections.

**On padding and leakage:** the convolution mixes neighbouring steps *inside the
window*. Every step in that window strictly precedes the target, so this is not
future leakage — the target row is never in its own window.

In [11]:
class CNNLSTM(nn.Module):
    """CNN-LSTM benchmark: conv feature extraction -> LSTM -> linear head."""

    def __init__(self, input_dim, n_targets, conv_channels=64, kernel_size=3,
                 lstm_hidden=128, lstm_layers=2, dropout=0.2):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(input_dim, conv_channels, kernel_size, padding=pad)
        self.conv2 = nn.Conv1d(conv_channels, conv_channels, kernel_size, padding=pad)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(conv_channels, lstm_hidden, num_layers=lstm_layers,
                            batch_first=True,
                            dropout=dropout if lstm_layers > 1 else 0.0,
                            bidirectional=False)
        self.head = nn.Linear(lstm_hidden, n_targets)

    def forward(self, x):                    # [B, T, F]
        h = x.transpose(1, 2)                # [B, F, T]
        h = self.act(self.conv1(h))
        h = self.act(self.conv2(h))
        h = self.drop(h).transpose(1, 2)     # [B, T, C]
        out, _ = self.lstm(h)
        return self.head(out[:, -1, :])      # [B, n_targets]


print("CNNLSTM defined.")


CNNLSTM defined.


## 10. TCN-LSTM — Paper Fidelity

```
Paper:   "Enhanced Multivariate Time Series Forecasting"
Authors: A. Mahmoud and A. Mohammed
Venue:   Neural Processing Letters, 56(5), p.223
Model:   TCN-LSTM
```

**Architecture described by the paper (high level):** a Temporal Convolutional
Network extracts temporal/local patterns → LSTM models sequential dependencies →
forecasting head.

**Implementation here:** a standard TCN residual stack — dilated **causal**
convolutions with `Chomp1d` cropping, weight normalisation, ReLU, dropout, and a
`1×1` residual projection when channel counts differ — then an LSTM and a linear
head.

`[B,T,F] → [TCN residual block × 4, dilations 1,2,4,8] → LSTM → last step → Linear → [B,2]`

**Implementation assumptions — not explicitly specified in the paper:**

- 4 residual blocks, 64 channels, kernel 3, dilations `1, 2, 4, 8`
  (receptive field `1 + 2·(3-1)·(1+2+4+8) = 61` ≥ `L = 56`, so the stack can see
  the whole window)
- Each block: 2 × (weight-normed causal Conv1d → Chomp → ReLU → Dropout) + residual
- LSTM: 2 layers, hidden 128, unidirectional; dropout 0.2; head `Linear(128 → 2)`

**Deliberately NOT added:** no Transformer, no attention, no bidirectional LSTM.
The convolutions are genuinely dilated and genuinely causal — this is not a
generic CNN.

**Causality:** `Chomp1d` strips the right-hand padding so output step `i` never
depends on input steps `> i`.

In [12]:
class Chomp1d(nn.Module):
    """Trim trailing padding to keep the convolution causal."""

    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x if self.chomp_size == 0 else x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """TCN residual block: 2x (causal dilated conv -> chomp -> ReLU -> dropout)."""

    def __init__(self, n_in, n_out, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = weight_norm(nn.Conv1d(n_in, n_out, kernel_size,
                                           padding=padding, dilation=dilation))
        self.chomp1, self.relu1, self.drop1 = Chomp1d(padding), nn.ReLU(), nn.Dropout(dropout)
        self.conv2 = weight_norm(nn.Conv1d(n_out, n_out, kernel_size,
                                           padding=padding, dilation=dilation))
        self.chomp2, self.relu2, self.drop2 = Chomp1d(padding), nn.ReLU(), nn.Dropout(dropout)
        self.downsample = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        self.relu_out = nn.ReLU()
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.drop1(self.relu1(self.chomp1(self.conv1(x))))
        out = self.drop2(self.relu2(self.chomp2(self.conv2(out))))
        res = x if self.downsample is None else self.downsample(x)
        return self.relu_out(out + res)


class TCNLSTM(nn.Module):
    """TCN-LSTM benchmark: dilated causal TCN -> LSTM -> linear head."""

    def __init__(self, input_dim, n_targets, tcn_channels=(64, 64, 64, 64),
                 kernel_size=3, lstm_hidden=128, lstm_layers=2, dropout=0.2):
        super().__init__()
        blocks, prev = [], input_dim
        for i, ch in enumerate(tcn_channels):
            blocks.append(TemporalBlock(prev, ch, kernel_size,
                                        dilation=2 ** i, dropout=dropout))
            prev = ch
        self.tcn = nn.Sequential(*blocks)
        self.lstm = nn.LSTM(prev, lstm_hidden, num_layers=lstm_layers,
                            batch_first=True,
                            dropout=dropout if lstm_layers > 1 else 0.0,
                            bidirectional=False)
        self.head = nn.Linear(lstm_hidden, n_targets)

    def forward(self, x):
        h = self.tcn(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.lstm(h)
        return self.head(out[:, -1, :])


_rf = 1 + 2 * (3 - 1) * (1 + 2 + 4 + 8)
print(f"TCNLSTM defined. Receptive field = {_rf} (sequence length L = {L}) "
      f"-> covers whole window: {_rf >= L}")


TCNLSTM defined. Receptive field = 61 (sequence length L = 56) -> covers whole window: True


## 11. LSTM-mTrans-MLP — Paper Fidelity

```
Paper:   "LSTM-Transformer-Based Robust Hybrid Deep Learning Model for
          Financial Time Series Forecasting"
Authors: Kabir et al.
Year:    2025
Venue:   Sci, 7(1), 7
Model:   LSTM-mTrans-MLP
```

**Architecture described by the paper (verified from the abstract):** a hybrid
integrating an **LSTM** network, a **modified Transformer** network, and a
**multilayered perceptron (MLP)** — LSTM for sequential dependencies, the
Transformer component for long-range temporal relationships, the MLP for
nonlinear feature relationships and final prediction.

**Implementation here:**
`[B,T,F] → LSTM → +positional encoding → mTrans blocks → mean-pool → MLP → Linear → [B,2]`

> **Implementation assumption — not explicitly specified in the paper.**
> The paper describes the "modified Transformer" conceptually, but I could not
> access implementation-level detail for the specific modification (the
> publisher page returned HTTP 403). This notebook implements it as a
> **pre-normalisation Transformer encoder** — LayerNorm before the attention
> sub-layer and before the feed-forward sub-layer, residual connections around
> both, GELU in the FFN — applied to the LSTM output sequence with sinusoidal
> positional encoding.
>
> This is a documented, reasonable reconstruction. It is **not** presented as the
> paper's exact mechanism. Given the paper's architecture subsection, this block
> can be revised to match precisely.

**Further implementation assumptions:** LSTM 2 layers / hidden 128 /
unidirectional; mTrans 2 blocks, `d_model=128`, 4 heads, FFN 256, dropout 0.2;
fixed sinusoidal positional encoding; mean pooling over time; MLP
`128 → 128 → 64 → 2` with GELU.

**Deliberately NOT added:** no CNN or TCN front end. The defining
`LSTM → mTrans → MLP` order is preserved.

**Attention masking:** self-attention is unmasked *within the window*. Every step
in the window precedes the target, so attending across it exposes no future
information relative to the prediction.

In [13]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)[:, :pe[:, 1::2].shape[1]]
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class MTransBlock(nn.Module):
    """Pre-norm Transformer encoder block.

    IMPLEMENTATION ASSUMPTION: pre-norm ordering + GELU FFN is this notebook's
    reconstruction of the paper's 'modified Transformer'; the paper's own
    implementation detail was not accessible.
    """

    def __init__(self, d_model, n_heads=4, ffn_dim=256, dropout=0.2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout,
                                          batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(ffn_dim, d_model))
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(a)
        return x + self.drop2(self.ffn(self.norm2(x)))


class LSTMmTransMLP(nn.Module):
    """LSTM -> modified Transformer -> MLP -> linear head."""

    def __init__(self, input_dim, n_targets, lstm_hidden=128, lstm_layers=2,
                 n_blocks=2, n_heads=4, ffn_dim=256,
                 mlp_hidden=(128, 64), dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, lstm_hidden, num_layers=lstm_layers,
                            batch_first=True,
                            dropout=dropout if lstm_layers > 1 else 0.0,
                            bidirectional=False)
        self.posenc = SinusoidalPositionalEncoding(lstm_hidden)
        self.blocks = nn.ModuleList([MTransBlock(lstm_hidden, n_heads, ffn_dim, dropout)
                                     for _ in range(n_blocks)])
        self.norm_out = nn.LayerNorm(lstm_hidden)
        layers, prev = [], lstm_hidden
        for hd in mlp_hidden:
            layers += [nn.Linear(prev, hd), nn.GELU(), nn.Dropout(dropout)]
            prev = hd
        self.mlp = nn.Sequential(*layers)
        self.head = nn.Linear(prev, n_targets)

    def forward(self, x):
        h, _ = self.lstm(x)
        h = self.posenc(h)
        for b in self.blocks:
            h = b(h)
        h = self.norm_out(h).mean(dim=1)
        return self.head(self.mlp(h))


print("LSTMmTransMLP defined.")


LSTMmTransMLP defined.


## 12. Shape validation, parameter counts, model summaries

In [14]:
def build_model(mk):
    if mk == "cnn_lstm":        return CNNLSTM(INPUT_DIM, N_TARGETS)
    if mk == "tcn_lstm":        return TCNLSTM(INPUT_DIM, N_TARGETS)
    if mk == "lstm_mtrans_mlp": return LSTMmTransMLP(INPUT_DIM, N_TARGETS)
    raise KeyError(mk)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


xb, yb = next(iter(train_loader))
B = xb.shape[0]
assert tuple(xb.shape) == (B, L, INPUT_DIM), xb.shape
assert tuple(yb.shape) == (B, N_TARGETS), yb.shape

MODEL_SUMMARY = {}
print("=" * 72)
print("SHAPE VALIDATION AND PARAMETER COUNTS")
print("=" * 72)
for mk in MODEL_KEYS:
    set_seed(RANDOM_SEED)
    m = build_model(mk).to(DEVICE).eval()
    with torch.no_grad():
        out = m(xb.to(DEVICE))
    if tuple(out.shape) != (B, N_TARGETS):
        raise ValueError(f"{MODEL_DISPLAY[mk]} -> {tuple(out.shape)}, expected {(B, N_TARGETS)}")
    if not torch.isfinite(out).all():
        raise ValueError(f"{MODEL_DISPLAY[mk]} produced non-finite output on the shape test.")
    n = count_params(m)
    MODEL_SUMMARY[mk] = {
        "model_name": MODEL_DISPLAY[mk],
        "input_shape": [L, INPUT_DIM], "output_shape": [N_TARGETS],
        "sequence_length": L, "n_features": INPUT_DIM,
        "trainable_parameters": int(n),
        "optimizer": "AdamW", "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE,
        "max_epochs": EPOCHS, "early_stopping_patience": PATIENCE,
        "scheduler": f"ReduceLROnPlateau(min, factor={SCHED_FACTOR}, "
                     f"patience={SCHED_PATIENCE}, min_lr={SCHED_MIN_LR})",
        "loss": "SmoothL1Loss", "device": str(DEVICE),
        "targets": TARGETS, "seed": RANDOM_SEED,
    }
    print(f"  {MODEL_DISPLAY[mk]:18s} Input [B,{L},{INPUT_DIM}] -> Output [B,{N_TARGETS}]  "
          f"PASSED | params={n:,}")
    del m

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

with open(OUT_P / "comparison" / "model_summary.json", "w") as f:
    json.dump(MODEL_SUMMARY, f, indent=2)
for mk in MODEL_KEYS:
    with open(mdir(mk, "config", "model_summary.json"), "w") as f:
        json.dump(MODEL_SUMMARY[mk], f, indent=2)

print("\n" + "=" * 72)
for mk in MODEL_KEYS:
    s = MODEL_SUMMARY[mk]
    print(s["model_name"])
    print(f"   Input {s['input_shape']} -> Output {s['output_shape']} | "
          f"seq={s['sequence_length']} feats={s['n_features']}")
    print(f"   params={s['trainable_parameters']:,} | {s['optimizer']} "
          f"lr={s['learning_rate']} wd={s['weight_decay']} bs={s['batch_size']}")
    print(f"   max_epochs={s['max_epochs']} patience={s['early_stopping_patience']} "
          f"loss={s['loss']} device={s['device']}")
    print(f"   scheduler={s['scheduler']}")
    print(f"   targets={s['targets']}")


SHAPE VALIDATION AND PARAMETER COUNTS


  CNN-LSTM           Input [B,56,39] -> Output [B,2]  PASSED | params=251,586


  TCN-LSTM           Input [B,56,39] -> Output [B,2]  PASSED | params=328,770


  LSTM-mTrans-MLP    Input [B,56,39] -> Output [B,2]  PASSED | params=508,738

CNN-LSTM
   Input [56, 39] -> Output [2] | seq=56 feats=39
   params=251,586 | AdamW lr=0.001 wd=0.0001 bs=128
   max_epochs=100 patience=15 loss=SmoothL1Loss device=cpu
   scheduler=ReduceLROnPlateau(min, factor=0.5, patience=5, min_lr=1e-07)
   targets=['NET_PATIENT_REVENUE', 'TOTAL_OPERATING_EXP']
TCN-LSTM
   Input [56, 39] -> Output [2] | seq=56 feats=39
   params=328,770 | AdamW lr=0.001 wd=0.0001 bs=128
   max_epochs=100 patience=15 loss=SmoothL1Loss device=cpu
   scheduler=ReduceLROnPlateau(min, factor=0.5, patience=5, min_lr=1e-07)
   targets=['NET_PATIENT_REVENUE', 'TOTAL_OPERATING_EXP']
LSTM-mTrans-MLP
   Input [56, 39] -> Output [2] | seq=56 feats=39
   params=508,738 | AdamW lr=0.001 wd=0.0001 bs=128
   max_epochs=100 patience=15 loss=SmoothL1Loss device=cpu
   scheduler=ReduceLROnPlateau(min, factor=0.5, patience=5, min_lr=1e-07)
   targets=['NET_PATIENT_REVENUE', 'TOTAL_OPERATING_EXP']


/usr/local/lib64/python3.9/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


## 13. Loss

`nn.SmoothL1Loss` (Huber) for **all three** models. Rationale: hospital financial
series carry occasional extreme values, and a Huber-type objective is less
dominated by a few large residuals than pure MSE while remaining a standard
regression loss.

> **Implementation decision, not paper-specified.** Neither paper's loss
> specification was accessible. Using the *same* loss across all three is
> required by the fair-comparison contract — the intended difference is the
> architecture, not the objective.

In [15]:
CRITERION = nn.SmoothL1Loss()
print("Loss:", CRITERION.__class__.__name__, "- identical for all three models")


Loss: SmoothL1Loss - identical for all three models


## 14. Checkpoint utilities

`latest.pt` (full resumable state, every epoch) and `best.pt` (lowest
**validation** loss). Compatibility is verified on load; a mismatch raises rather
than loading wrong weights. Writes are atomic (temp file + replace) so an
interrupt cannot corrupt a good checkpoint. **The test set never influences
checkpoint selection.**

In [16]:
def _sig(mk):
    return {"model_name": MODEL_DISPLAY[mk], "input_dim": int(INPUT_DIM),
            "sequence_length": int(L), "n_targets": int(N_TARGETS)}


def save_latest(mk, epoch, model, opt, sched, scaler, best_vl, best_ep, hist, cfg):
    p = mdir(mk, "checkpoints", "latest", "latest.pt")
    payload = {"epoch": int(epoch), "model_state_dict": model.state_dict(),
               "optimizer_state_dict": opt.state_dict(),
               "scheduler_state_dict": sched.state_dict(),
               "best_val_loss": float(best_vl), "best_epoch": int(best_ep),
               "history": hist, "config": cfg, "compat": _sig(mk)}
    if scaler is not None:
        payload["scaler_state_dict"] = scaler.state_dict()
    tmp = p.with_suffix(".pt.tmp"); torch.save(payload, tmp); tmp.replace(p)


def save_best(mk, epoch, model, vl, hist, cfg):
    p = mdir(mk, "checkpoints", "best", "best.pt")
    tmp = p.with_suffix(".pt.tmp")
    torch.save({"epoch": int(epoch), "model_state_dict": model.state_dict(),
                "val_loss": float(vl), "history": hist, "config": cfg,
                "compat": _sig(mk)}, tmp)
    tmp.replace(p)
    with open(mdir(mk, "checkpoints", "best", "best_model_info.json"), "w") as f:
        json.dump({"model_name": MODEL_DISPLAY[mk], "best_epoch": int(epoch),
                   "best_validation_loss": float(vl)}, f, indent=2)


def _check_compat(mk, ck, path):
    want, got = _sig(mk), ck.get("compat")
    if got is None:
        raise ValueError(f"Checkpoint {path} has no compatibility signature. "
                         "Delete it or set RESUME=False.")
    if got != want:
        raise ValueError("Checkpoint is incompatible with the current dataset/model "
                         f"configuration.\n  checkpoint: {got}\n  current   : {want}\n"
                         f"  file: {path}\nRefusing to load mismatched weights.")


def load_latest(mk, model, opt, sched, scaler):
    p = mdir(mk, "checkpoints", "latest", "latest.pt")
    if not p.exists():
        return None
    try:
        ck = torch.load(p, map_location=DEVICE, weights_only=False)
    except Exception as e:
        raise RuntimeError(f"Could not read checkpoint {p}: {e}\n"
                           "It may be corrupted. Delete it or set RESUME=False.") from e
    _check_compat(mk, ck, p)
    model.load_state_dict(ck["model_state_dict"])
    opt.load_state_dict(ck["optimizer_state_dict"])
    sched.load_state_dict(ck["scheduler_state_dict"])
    if scaler is not None and "scaler_state_dict" in ck:
        scaler.load_state_dict(ck["scaler_state_dict"])
    return (int(ck["epoch"]) + 1, float(ck["best_val_loss"]),
            int(ck.get("best_epoch", ck["epoch"])), list(ck.get("history", [])))


def load_best_for_eval(mk):
    p = mdir(mk, "checkpoints", "best", "best.pt")
    if not p.exists():
        raise FileNotFoundError(
            f"No best checkpoint for {MODEL_DISPLAY[mk]} at {p}. Train first. "
            "The latest checkpoint is deliberately NOT substituted for final reporting.")
    ck = torch.load(p, map_location=DEVICE, weights_only=False)
    _check_compat(mk, ck, p)
    m = build_model(mk).to(DEVICE)
    m.load_state_dict(ck["model_state_dict"]); m.eval()
    return m, int(ck["epoch"]), float(ck["val_loss"])


print("Checkpoint utilities ready (atomic writes, compat-verified, resumable).")


Checkpoint utilities ready (atomic writes, compat-verified, resumable).


## 15. Training engine

Per epoch: all **ten** metrics for **both** targets on **train** and
**validation** (40 metric columns), plus `train_loss`, `val_loss`,
`learning_rate`, `epoch_time`. Includes a non-finite-loss guard with
diagnostics, optional gradient clipping, and duplicate-epoch protection when
appending history on resume.

In [17]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    losses, ys, ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            out = model(xb); loss = CRITERION(out, yb)
        losses.append(float(loss.detach().float().cpu()))
        ys.append(yb.detach().float().cpu().numpy())
        ps.append(out.detach().float().cpu().numpy())
    return float(np.mean(losses)), np.concatenate(ys), np.concatenate(ps)


def report_epoch(name, ep, tot, tl, vl, lr, trm, vam, secs):
    print("=" * 60)
    print(f"{name} | Epoch {ep}/{tot}")
    print("=" * 60)
    print(f"\nTrain Loss: {tl:.6f}\nVal Loss:   {vl:.6f}\nLR:         {lr:.8f}"
          f"\nEpoch time: {secs:.1f}s")
    for t in TARGETS:
        for lbl, d in (("TRAIN", trm), ("VALIDATION", vam)):
            print(f"\n{t} - {lbl}")
            for k in METRIC_NAMES:
                print(f"{k}: {d[t][k]:.6f}")
    print()


def train_model(mk, resume=None):
    name = MODEL_DISPLAY[mk]
    resume = RESUME if resume is None else resume
    set_seed(RANDOM_SEED)

    model = build_model(mk).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                            weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=SCHED_FACTOR,
        patience=SCHED_PATIENCE, min_lr=SCHED_MIN_LR)
    scaler = torch.amp.GradScaler(DEVICE.type) if AMP_ENABLED else None

    cfg = dict(MODEL_SUMMARY[mk])
    cfg.update({
        "resume_requested": bool(resume), "amp_enabled": bool(AMP_ENABLED),
        "grad_clip_norm": GRAD_CLIP_NORM, "mape_epsilon": MAPE_EPSILON,
        "train_metrics_eval_pass": bool(TRAIN_METRICS_EVAL_PASS),
        "target_transform": "log1p + train-fit z-score (supplied by preprocessing)",
        "train_target_std_original_units": TRAIN_TARGET_STD.tolist(),
        "feature_columns": FEATURE_COLS,
        "n_train_windows": int(len(X_train)), "n_val_windows": int(len(X_val)),
        "n_test_windows": int(len(X_test)),
        "implementation_notes": [
            "Layer sizes are implementation assumptions, not paper-specified.",
            "SmoothL1Loss shared across all three models (implementation decision).",
            "AdamW + ReduceLROnPlateau are implementation configuration.",
            "Gradient clipping is a numerical-stability measure, not paper-specified.",
        ],
    })
    with open(mdir(mk, "config", "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    start_ep, best_vl, best_ep, hist = 1, float("inf"), -1, []
    if resume:
        got = load_latest(mk, model, opt, sched, scaler)
        if got is None:
            print(f"[{name}] RESUME=True but no checkpoint found - starting at epoch 1.")
        else:
            start_ep, best_vl, best_ep, hist = got
            print(f"[{name}] Resumed. Continuing at epoch {start_ep} "
                  f"(best val {best_vl:.6f} @ epoch {best_ep}).")
    if start_ep > EPOCHS:
        print(f"[{name}] Already at epoch {start_ep-1} >= EPOCHS={EPOCHS}. Nothing to do.")
        return hist

    no_improve = len([h for h in hist if h["epoch"] > best_ep]) if hist else 0

    for ep in range(start_ep, EPOCHS + 1):
        t0 = time.time()
        model.train()
        blosses, trt, trp = [], [], []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                out = model(xb); loss = CRITERION(out, yb)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"[{name}] Non-finite loss at epoch {ep} (loss={loss.item()}).\n"
                    f"  finite inputs : {bool(torch.isfinite(xb).all())}\n"
                    f"  finite targets: {bool(torch.isfinite(yb).all())}\n"
                    f"  max|input|    : {float(xb.abs().max()):.4f}\n"
                    f"  AMP enabled   : {AMP_ENABLED}\n"
                    "  Likely causes: unscaled inputs, learning rate too high, or fp16 "
                    "overflow under AMP. Values are not silently replaced.")

            if scaler is not None:
                scaler.scale(loss).backward()
                if GRAD_CLIP_NORM is not None:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                opt.step()

            blosses.append(float(loss.detach().float().cpu()))
            if not TRAIN_METRICS_EVAL_PASS:
                trt.append(yb.detach().float().cpu().numpy())
                trp.append(out.detach().float().cpu().numpy())

        tl = float(np.mean(blosses))
        if TRAIN_METRICS_EVAL_PASS:
            _, tra, trb = evaluate(model, train_loader)
        else:
            tra, trb = np.concatenate(trt), np.concatenate(trp)
        trm = metrics_all(tra, trb)

        vl, vat, vap = evaluate(model, val_loader)
        vam = metrics_all(vat, vap)

        sched.step(vl)
        lr = float(opt.param_groups[0]["lr"])
        secs = time.time() - t0

        row = {"epoch": int(ep), "learning_rate": lr, "train_loss": tl,
               "val_loss": vl, "epoch_time": secs}
        row.update(flatten("train", trm)); row.update(flatten("val", vam))
        hist = [h for h in hist if int(h["epoch"]) != int(ep)]
        hist.append(row); hist.sort(key=lambda h: int(h["epoch"]))
        pd.DataFrame(hist).to_csv(mdir(mk, "metrics", "history.csv"), index=False)

        report_epoch(name, ep, EPOCHS, tl, vl, lr, trm, vam, secs)
        with open(mdir(mk, "logs", "train_log.txt"), "a") as lf:
            lf.write(f"epoch={ep} train_loss={tl:.6f} val_loss={vl:.6f} "
                     f"lr={lr:.8f} time={secs:.1f}s\n")

        if vl < best_vl - 1e-12:
            best_vl, best_ep, no_improve = vl, ep, 0
            save_best(mk, ep, model, vl, hist, cfg)
            print(f"  -> new best validation loss ({best_vl:.6f}); best.pt updated")
        else:
            no_improve += 1
            print(f"  -> no improvement {no_improve}/{PATIENCE} "
                  f"(best {best_vl:.6f} @ epoch {best_ep})")

        save_latest(mk, ep, model, opt, sched, scaler, best_vl, best_ep, hist, cfg)

        if no_improve >= PATIENCE:
            print("\nEarly stopping triggered.")
            print(f"Best epoch: {best_ep}")
            print(f"Best validation loss: {best_vl:.6f}")
            break

    print(f"\n[{name}] Done. Best epoch {best_ep}, best val loss {best_vl:.6f}")
    return hist


print("Training engine ready.")


Training engine ready.


## 16. CNN-LSTM — training

In [18]:
HISTORIES = {}
if "cnn_lstm" in MODELS_TO_RUN:
    HISTORIES["cnn_lstm"] = train_model("cnn_lstm")
else:
    print("cnn_lstm skipped.")


[CNN-LSTM] RESUME=True but no checkpoint found - starting at epoch 1.


CNN-LSTM | Epoch 1/100

Train Loss: 0.303020
Val Loss:   0.240884
LR:         0.00100000
Epoch time: 1.5s

NET_PATIENT_REVENUE - TRAIN
MAE: 14000.830013
RMSE: 17265.595364
MAPE: 10.059157
sMAPE: 9.857823
R2: 0.286727
ExplainedVar: 0.287368
MedianAE: 12287.718152
MaxError: 60648.485334
MAE_scaled: 0.684857
RMSE_scaled: 0.844555

NET_PATIENT_REVENUE - VALIDATION
MAE: 15646.853980
RMSE: 18154.207398
MAPE: 8.955895
sMAPE: 9.166649
R2: -0.039758
ExplainedVar: 0.150646
MedianAE: 15123.112529
MaxError: 53167.150257
MAE_scaled: 0.765374
RMSE_scaled: 0.888022

TOTAL_OPERATING_EXP - TRAIN
MAE: 5013.953183
RMSE: 6429.498950
MAPE: 5.004758
sMAPE: 4.985335
R2: 0.285166
ExplainedVar: 0.285663
MedianAE: 4330.082058
MaxError: 48240.901297
MAE_scaled: 0.659334
RMSE_scaled: 0.845479

TOTAL_OPERATING_EXP - VALIDATION
MAE: 4822.863011
RMSE: 5699.071517
MAPE: 4.349650
sMAPE: 4.388534
R2: 0.063342
ExplainedVar: 0.162563
MedianAE: 4807.935771
MaxError: 31357.387761
MAE_scaled: 0.634206
RMSE_scaled: 0.749427


CNN-LSTM | Epoch 2/100

Train Loss: 0.195549
Val Loss:   0.139096
LR:         0.00100000
Epoch time: 1.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 11153.509646
RMSE: 13636.528383
MAPE: 8.080902
sMAPE: 7.897679
R2: 0.555061
ExplainedVar: 0.555085
MedianAE: 9914.061976
MaxError: 45684.072818
MAE_scaled: 0.545579
RMSE_scaled: 0.667037

NET_PATIENT_REVENUE - VALIDATION
MAE: 11227.350145
RMSE: 13860.813699
MAPE: 6.724562
sMAPE: 6.600127
R2: 0.393885
ExplainedVar: 0.396624
MedianAE: 9943.949070
MaxError: 51880.735925
MAE_scaled: 0.549191
RMSE_scaled: 0.678008

TOTAL_OPERATING_EXP - TRAIN
MAE: 3811.567067
RMSE: 5036.504080
MAPE: 3.810765
sMAPE: 3.792579
R2: 0.561359
ExplainedVar: 0.561380
MedianAE: 3302.894564
MaxError: 42642.259174
MAE_scaled: 0.501221
RMSE_scaled: 0.662300

TOTAL_OPERATING_EXP - VALIDATION
MAE: 3354.798791
RMSE: 4560.548836
MAPE: 3.105065
sMAPE: 3.065496
R2: 0.400199
ExplainedVar: 0.425414
MedianAE: 2551.879669
MaxError: 26049.241611
MAE_scaled: 0.441156
RMSE_scaled: 0.599712

  -

CNN-LSTM | Epoch 3/100

Train Loss: 0.130298
Val Loss:   0.117946
LR:         0.00100000
Epoch time: 1.5s

NET_PATIENT_REVENUE - TRAIN
MAE: 8929.196878
RMSE: 11118.571157
MAPE: 6.384814
sMAPE: 6.320411
R2: 0.704205
ExplainedVar: 0.704232
MedianAE: 7709.871910
MaxError: 37467.275020
MAE_scaled: 0.436776
RMSE_scaled: 0.543870

NET_PATIENT_REVENUE - VALIDATION
MAE: 10550.841488
RMSE: 13133.171054
MAPE: 5.987753
sMAPE: 6.129505
R2: 0.455852
ExplainedVar: 0.553013
MedianAE: 9037.873575
MaxError: 41415.550102
MAE_scaled: 0.516100
RMSE_scaled: 0.642415

TOTAL_OPERATING_EXP - TRAIN
MAE: 2984.626936
RMSE: 4208.428099
MAPE: 2.953404
sMAPE: 2.962430
R2: 0.693740
ExplainedVar: 0.693828
MedianAE: 2486.463034
MaxError: 42625.272882
MAE_scaled: 0.392478
RMSE_scaled: 0.553408

TOTAL_OPERATING_EXP - VALIDATION
MAE: 3142.329819
RMSE: 4131.557748
MAPE: 2.816950
sMAPE: 2.845767
R2: 0.507733
ExplainedVar: 0.562596
MedianAE: 2728.405709
MaxError: 27318.034427
MAE_scaled: 0.413216
RMSE_scaled: 0.543299

  ->

CNN-LSTM | Epoch 4/100

Train Loss: 0.099467
Val Loss:   0.104076
LR:         0.00100000
Epoch time: 1.4s

NET_PATIENT_REVENUE - TRAIN
MAE: 7695.104242
RMSE: 9737.415944
MAPE: 5.450528
sMAPE: 5.414141
R2: 0.773128
ExplainedVar: 0.773130
MedianAE: 6372.108901
MaxError: 38822.115545
MAE_scaled: 0.376410
RMSE_scaled: 0.476310

NET_PATIENT_REVENUE - VALIDATION
MAE: 9566.927030
RMSE: 11813.485495
MAPE: 5.561535
sMAPE: 5.563598
R2: 0.559715
ExplainedVar: 0.568865
MedianAE: 8529.491797
MaxError: 38972.088693
MAE_scaled: 0.467971
RMSE_scaled: 0.577862

TOTAL_OPERATING_EXP - TRAIN
MAE: 2498.828815
RMSE: 3802.316321
MAPE: 2.458469
sMAPE: 2.469811
R2: 0.749996
ExplainedVar: 0.750185
MedianAE: 1963.003810
MaxError: 40952.506688
MAE_scaled: 0.328596
RMSE_scaled: 0.500004

TOTAL_OPERATING_EXP - VALIDATION
MAE: 2867.762504
RMSE: 3925.584535
MAPE: 2.615426
sMAPE: 2.605679
R2: 0.555592
ExplainedVar: 0.556081
MedianAE: 2320.501133
MaxError: 26650.201031
MAE_scaled: 0.377111
RMSE_scaled: 0.516214

  -> n

...

CNN-LSTM | Epoch 27/100

Train Loss: 0.048517
Val Loss:   0.056429
LR:         0.00050000
Epoch time: 1.7s

NET_PATIENT_REVENUE - TRAIN
MAE: 5365.253101
RMSE: 6874.017636
MAPE: 3.774917
sMAPE: 3.762060
R2: 0.886939
ExplainedVar: 0.886942
MedianAE: 4495.872304
MaxError: 35810.011599
MAE_scaled: 0.262444
RMSE_scaled: 0.336246

NET_PATIENT_REVENUE - VALIDATION
MAE: 7385.096830
RMSE: 9290.948881
MAPE: 4.304744
sMAPE: 4.318987
R2: 0.727668
ExplainedVar: 0.731895
MedianAE: 5974.290472
MaxError: 36228.682798
MAE_scaled: 0.361246
RMSE_scaled: 0.454471

TOTAL_OPERATING_EXP - TRAIN
MAE: 1415.513594
RMSE: 2872.077331
MAPE: 1.376769
sMAPE: 1.390857
R2: 0.857359
ExplainedVar: 0.857801
MedianAE: 966.065846
MaxError: 38807.132581
MAE_scaled: 0.186140
RMSE_scaled: 0.377678

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1672.137326
RMSE: 3007.526494
MAPE: 1.510262
sMAPE: 1.518286
R2: 0.739149
ExplainedVar: 0.739374
MedianAE: 1114.536792
MaxError: 28723.995939
MAE_scaled: 0.219886
RMSE_scaled: 0.395489

  -> no

CNN-LSTM | Epoch 28/100

Train Loss: 0.048610
Val Loss:   0.063476
LR:         0.00025000
Epoch time: 1.4s

NET_PATIENT_REVENUE - TRAIN
MAE: 5386.785780
RMSE: 6873.037434
MAPE: 3.776763
sMAPE: 3.767564
R2: 0.886971
ExplainedVar: 0.887051
MedianAE: 4533.155816
MaxError: 38363.891251
MAE_scaled: 0.263497
RMSE_scaled: 0.336198

NET_PATIENT_REVENUE - VALIDATION
MAE: 7827.220891
RMSE: 9829.956822
MAPE: 4.517058
sMAPE: 4.575892
R2: 0.695154
ExplainedVar: 0.723303
MedianAE: 6523.379436
MaxError: 38112.382934
MAE_scaled: 0.382872
RMSE_scaled: 0.480837

TOTAL_OPERATING_EXP - TRAIN
MAE: 1423.726626
RMSE: 2854.307617
MAPE: 1.384577
sMAPE: 1.397293
R2: 0.859119
ExplainedVar: 0.859259
MedianAE: 985.070662
MaxError: 37811.219158
MAE_scaled: 0.187220
RMSE_scaled: 0.375341

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1838.959286
RMSE: 3126.267721
MAPE: 1.649797
sMAPE: 1.667866
R2: 0.718145
ExplainedVar: 0.737366
MedianAE: 1435.076157
MaxError: 29949.812405
MAE_scaled: 0.241823
RMSE_scaled: 0.411104

  -> no

CNN-LSTM | Epoch 29/100

Train Loss: 0.047907
Val Loss:   0.060274
LR:         0.00025000
Epoch time: 1.4s

NET_PATIENT_REVENUE - TRAIN
MAE: 5353.606165
RMSE: 6820.048769
MAPE: 3.760511
sMAPE: 3.750957
R2: 0.888707
ExplainedVar: 0.888834
MedianAE: 4518.893397
MaxError: 36722.524658
MAE_scaled: 0.261874
RMSE_scaled: 0.333606

NET_PATIENT_REVENUE - VALIDATION
MAE: 7520.160607
RMSE: 9391.561630
MAPE: 4.422592
sMAPE: 4.392405
R2: 0.721738
ExplainedVar: 0.721966
MedianAE: 6286.885818
MaxError: 35220.500826
MAE_scaled: 0.367852
RMSE_scaled: 0.459393

TOTAL_OPERATING_EXP - TRAIN
MAE: 1421.466381
RMSE: 2863.291425
MAPE: 1.381347
sMAPE: 1.395199
R2: 0.858231
ExplainedVar: 0.858844
MedianAE: 1004.734321
MaxError: 38967.258095
MAE_scaled: 0.186923
RMSE_scaled: 0.376523

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1786.624496
RMSE: 3072.783360
MAPE: 1.616677
sMAPE: 1.620467
R2: 0.727707
ExplainedVar: 0.727832
MedianAE: 1261.836397
MaxError: 27698.647547
MAE_scaled: 0.234941
RMSE_scaled: 0.404071

  -> n

CNN-LSTM | Epoch 30/100

Train Loss: 0.046709
Val Loss:   0.058802
LR:         0.00025000
Epoch time: 1.6s

NET_PATIENT_REVENUE - TRAIN
MAE: 5261.666505
RMSE: 6749.239524
MAPE: 3.690248
sMAPE: 3.680535
R2: 0.891006
ExplainedVar: 0.891066
MedianAE: 4343.869902
MaxError: 35866.572821
MAE_scaled: 0.257377
RMSE_scaled: 0.330142

NET_PATIENT_REVENUE - VALIDATION
MAE: 7499.165907
RMSE: 9360.011702
MAPE: 4.417306
sMAPE: 4.382532
R2: 0.723605
ExplainedVar: 0.724585
MedianAE: 6380.025011
MaxError: 35239.291670
MAE_scaled: 0.366825
RMSE_scaled: 0.457850

TOTAL_OPERATING_EXP - TRAIN
MAE: 1380.073475
RMSE: 2833.721223
MAPE: 1.341966
sMAPE: 1.354494
R2: 0.861144
ExplainedVar: 0.861267
MedianAE: 948.242376
MaxError: 38410.524730
MAE_scaled: 0.181480
RMSE_scaled: 0.372634

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1760.716807
RMSE: 3050.919635
MAPE: 1.587996
sMAPE: 1.595374
R2: 0.731568
ExplainedVar: 0.732165
MedianAE: 1267.297583
MaxError: 28006.667572
MAE_scaled: 0.231534
RMSE_scaled: 0.401196

  -> no

CNN-LSTM | Epoch 31/100

Train Loss: 0.046612
Val Loss:   0.058628
LR:         0.00025000
Epoch time: 1.6s

NET_PATIENT_REVENUE - TRAIN
MAE: 5279.116434
RMSE: 6773.832570
MAPE: 3.710003
sMAPE: 3.693650
R2: 0.890210
ExplainedVar: 0.890235
MedianAE: 4396.640101
MaxError: 37504.485150
MAE_scaled: 0.258231
RMSE_scaled: 0.331345

NET_PATIENT_REVENUE - VALIDATION
MAE: 7551.692018
RMSE: 9438.512795
MAPE: 4.395776
sMAPE: 4.405470
R2: 0.718949
ExplainedVar: 0.723495
MedianAE: 6027.781414
MaxError: 37591.269380
MAE_scaled: 0.369395
RMSE_scaled: 0.461689

TOTAL_OPERATING_EXP - TRAIN
MAE: 1401.663413
RMSE: 2837.195905
MAPE: 1.362661
sMAPE: 1.376146
R2: 0.860803
ExplainedVar: 0.861140
MedianAE: 954.549191
MaxError: 37775.095818
MAE_scaled: 0.184319
RMSE_scaled: 0.373091

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1725.830360
RMSE: 3045.821933
MAPE: 1.558656
sMAPE: 1.564504
R2: 0.732464
ExplainedVar: 0.732474
MedianAE: 1174.259536
MaxError: 28274.691954
MAE_scaled: 0.226947
RMSE_scaled: 0.400525

  -> no

## 17. TCN-LSTM — training

In [19]:
if "tcn_lstm" in MODELS_TO_RUN:
    HISTORIES["tcn_lstm"] = train_model("tcn_lstm")
else:
    print("tcn_lstm skipped.")


[TCN-LSTM] RESUME=True but no checkpoint found - starting at epoch 1.


/usr/local/lib64/python3.9/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


TCN-LSTM | Epoch 1/100

Train Loss: 0.287110
Val Loss:   0.267711
LR:         0.00100000
Epoch time: 2.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 13652.411266
RMSE: 16704.064912
MAPE: 9.809926
sMAPE: 9.611827
R2: 0.332368
ExplainedVar: 0.332546
MedianAE: 12053.924587
MaxError: 57337.754306
MAE_scaled: 0.667814
RMSE_scaled: 0.817088

NET_PATIENT_REVENUE - VALIDATION
MAE: 16803.370189
RMSE: 19568.263949
MAPE: 9.512672
sMAPE: 9.864137
R2: -0.208043
ExplainedVar: 0.126164
MedianAE: 16219.841144
MaxError: 54194.198799
MAE_scaled: 0.821945
RMSE_scaled: 0.957191

TOTAL_OPERATING_EXP - TRAIN
MAE: 4874.786257
RMSE: 6230.057237
MAPE: 4.871948
sMAPE: 4.845110
R2: 0.328826
ExplainedVar: 0.328828
MedianAE: 4251.716457
MaxError: 42604.767236
MAE_scaled: 0.641034
RMSE_scaled: 0.819252

TOTAL_OPERATING_EXP - VALIDATION
MAE: 4964.267282
RMSE: 5784.688955
MAPE: 4.466683
sMAPE: 4.519178
R2: 0.034987
ExplainedVar: 0.175882
MedianAE: 4893.946053
MaxError: 31546.158135
MAE_scaled: 0.652801
RMSE_scaled: 0.760686



TCN-LSTM | Epoch 2/100

Train Loss: 0.196483
Val Loss:   0.145416
LR:         0.00100000
Epoch time: 2.4s

NET_PATIENT_REVENUE - TRAIN
MAE: 11158.911950
RMSE: 13686.010456
MAPE: 8.097464
sMAPE: 7.897463
R2: 0.551826
ExplainedVar: 0.552110
MedianAE: 9857.557475
MaxError: 50837.499946
MAE_scaled: 0.545844
RMSE_scaled: 0.669458

NET_PATIENT_REVENUE - VALIDATION
MAE: 12219.898202
RMSE: 14807.098707
MAPE: 6.933164
sMAPE: 7.128078
R2: 0.308300
ExplainedVar: 0.469864
MedianAE: 11193.431419
MaxError: 47803.963372
MAE_scaled: 0.597742
RMSE_scaled: 0.724296

TOTAL_OPERATING_EXP - TRAIN
MAE: 3860.411356
RMSE: 5027.430124
MAPE: 3.860994
sMAPE: 3.843324
R2: 0.562938
ExplainedVar: 0.563145
MedianAE: 3391.773187
MaxError: 43006.191885
MAE_scaled: 0.507644
RMSE_scaled: 0.661107

TOTAL_OPERATING_EXP - VALIDATION
MAE: 3548.752103
RMSE: 4478.737849
MAPE: 3.177441
sMAPE: 3.221049
R2: 0.421525
ExplainedVar: 0.527266
MedianAE: 3234.083457
MaxError: 27254.579146
MAE_scaled: 0.466661
RMSE_scaled: 0.588954

  

TCN-LSTM | Epoch 3/100

Train Loss: 0.132930
Val Loss:   0.123036
LR:         0.00100000
Epoch time: 2.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 9081.466951
RMSE: 11271.521471
MAPE: 6.499221
sMAPE: 6.429002
R2: 0.696011
ExplainedVar: 0.696012
MedianAE: 7742.075082
MaxError: 42115.427587
MAE_scaled: 0.444224
RMSE_scaled: 0.551352

NET_PATIENT_REVENUE - VALIDATION
MAE: 10185.932706
RMSE: 12544.158980
MAPE: 5.846646
sMAPE: 5.919477
R2: 0.503567
ExplainedVar: 0.546242
MedianAE: 8620.066269
MaxError: 42173.159601
MAE_scaled: 0.498250
RMSE_scaled: 0.613604

TOTAL_OPERATING_EXP - TRAIN
MAE: 3016.584206
RMSE: 4225.185398
MAPE: 2.987469
sMAPE: 2.994536
R2: 0.691296
ExplainedVar: 0.691356
MedianAE: 2543.301017
MaxError: 41208.433694
MAE_scaled: 0.396681
RMSE_scaled: 0.555612

TOTAL_OPERATING_EXP - VALIDATION
MAE: 3393.230664
RMSE: 4332.753132
MAPE: 3.057687
sMAPE: 3.074850
R2: 0.458622
ExplainedVar: 0.483542
MedianAE: 3221.771404
MaxError: 28525.192972
MAE_scaled: 0.446210
RMSE_scaled: 0.569757

  ->

...

TCN-LSTM | Epoch 40/100

Train Loss: 0.049165
Val Loss:   0.055676
LR:         0.00012500
Epoch time: 2.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 5459.749414
RMSE: 6993.687603
MAPE: 3.848484
sMAPE: 3.834248
R2: 0.882968
ExplainedVar: 0.882988
MedianAE: 4430.695410
MaxError: 37977.224113
MAE_scaled: 0.267066
RMSE_scaled: 0.342100

NET_PATIENT_REVENUE - VALIDATION
MAE: 7129.159879
RMSE: 9079.129086
MAPE: 4.112628
sMAPE: 4.145666
R2: 0.739944
ExplainedVar: 0.753545
MedianAE: 5794.954771
MaxError: 35618.438221
MAE_scaled: 0.348726
RMSE_scaled: 0.444110

TOTAL_OPERATING_EXP - TRAIN
MAE: 1432.924028
RMSE: 2890.038872
MAPE: 1.393960
sMAPE: 1.407446
R2: 0.855570
ExplainedVar: 0.855910
MedianAE: 1002.368035
MaxError: 37200.612900
MAE_scaled: 0.188429
RMSE_scaled: 0.380040

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1692.816093
RMSE: 2981.762561
MAPE: 1.519620
sMAPE: 1.532091
R2: 0.743599
ExplainedVar: 0.750977
MedianAE: 1186.687212
MaxError: 27087.572165
MAE_scaled: 0.222605
RMSE_scaled: 0.392102

  -> n

TCN-LSTM | Epoch 41/100

Train Loss: 0.050065
Val Loss:   0.055958
LR:         0.00006250
Epoch time: 2.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 5454.497101
RMSE: 6956.640456
MAPE: 3.840446
sMAPE: 3.825635
R2: 0.884204
ExplainedVar: 0.884230
MedianAE: 4569.013355
MaxError: 35745.461193
MAE_scaled: 0.266809
RMSE_scaled: 0.340287

NET_PATIENT_REVENUE - VALIDATION
MAE: 7112.342133
RMSE: 9054.299291
MAPE: 4.105734
sMAPE: 4.136655
R2: 0.741365
ExplainedVar: 0.753704
MedianAE: 5749.962390
MaxError: 35536.369214
MAE_scaled: 0.347904
RMSE_scaled: 0.442895

TOTAL_OPERATING_EXP - TRAIN
MAE: 1442.276647
RMSE: 2897.952730
MAPE: 1.402561
sMAPE: 1.415311
R2: 0.854778
ExplainedVar: 0.854954
MedianAE: 999.175447
MaxError: 38428.648791
MAE_scaled: 0.189659
RMSE_scaled: 0.381081

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1724.151965
RMSE: 3005.416439
MAPE: 1.545925
sMAPE: 1.559908
R2: 0.739515
ExplainedVar: 0.750555
MedianAE: 1204.987038
MaxError: 27191.938395
MAE_scaled: 0.226726
RMSE_scaled: 0.395212

  -> no

TCN-LSTM | Epoch 42/100

Train Loss: 0.049054
Val Loss:   0.054086
LR:         0.00006250
Epoch time: 2.2s

NET_PATIENT_REVENUE - TRAIN
MAE: 5439.669320
RMSE: 6949.801733
MAPE: 3.822866
sMAPE: 3.812677
R2: 0.884432
ExplainedVar: 0.884652
MedianAE: 4480.732016
MaxError: 36638.065572
MAE_scaled: 0.266084
RMSE_scaled: 0.339953

NET_PATIENT_REVENUE - VALIDATION
MAE: 7014.460463
RMSE: 8922.001128
MAPE: 4.070026
sMAPE: 4.081511
R2: 0.748868
ExplainedVar: 0.753251
MedianAE: 5703.988261
MaxError: 34611.461856
MAE_scaled: 0.343116
RMSE_scaled: 0.436424

TOTAL_OPERATING_EXP - TRAIN
MAE: 1446.339990
RMSE: 2915.287229
MAPE: 1.406126
sMAPE: 1.419998
R2: 0.853035
ExplainedVar: 0.853526
MedianAE: 1019.624196
MaxError: 38179.045594
MAE_scaled: 0.190194
RMSE_scaled: 0.383360

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1669.674310
RMSE: 2970.686599
MAPE: 1.500755
sMAPE: 1.511105
R2: 0.745501
ExplainedVar: 0.749218
MedianAE: 1187.490700
MaxError: 26987.197640
MAE_scaled: 0.219562
RMSE_scaled: 0.390645

  -> n

TCN-LSTM | Epoch 43/100

Train Loss: 0.048922
Val Loss:   0.055484
LR:         0.00006250
Epoch time: 2.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 5429.349928
RMSE: 6922.353722
MAPE: 3.826141
sMAPE: 3.810300
R2: 0.885343
ExplainedVar: 0.885347
MedianAE: 4556.652336
MaxError: 37383.085160
MAE_scaled: 0.265579
RMSE_scaled: 0.338610

NET_PATIENT_REVENUE - VALIDATION
MAE: 7094.442176
RMSE: 9028.722636
MAPE: 4.098754
sMAPE: 4.128967
R2: 0.742824
ExplainedVar: 0.754384
MedianAE: 5674.024495
MaxError: 35216.694930
MAE_scaled: 0.347028
RMSE_scaled: 0.441644

TOTAL_OPERATING_EXP - TRAIN
MAE: 1436.987873
RMSE: 2895.119144
MAPE: 1.397619
sMAPE: 1.410694
R2: 0.855061
ExplainedVar: 0.855314
MedianAE: 980.945975
MaxError: 38086.712059
MAE_scaled: 0.188964
RMSE_scaled: 0.380708

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1705.418940
RMSE: 2992.901766
MAPE: 1.529739
sMAPE: 1.543395
R2: 0.741680
ExplainedVar: 0.751735
MedianAE: 1183.749939
MaxError: 27186.092374
MAE_scaled: 0.224262
RMSE_scaled: 0.393566

  -> no

TCN-LSTM | Epoch 44/100

Train Loss: 0.048973
Val Loss:   0.054353
LR:         0.00006250
Epoch time: 2.2s

NET_PATIENT_REVENUE - TRAIN
MAE: 5396.459415
RMSE: 6906.512738
MAPE: 3.797546
sMAPE: 3.781841
R2: 0.885867
ExplainedVar: 0.885871
MedianAE: 4438.685398
MaxError: 37062.145387
MAE_scaled: 0.263970
RMSE_scaled: 0.337835

NET_PATIENT_REVENUE - VALIDATION
MAE: 7041.085000
RMSE: 8957.885316
MAPE: 4.076155
sMAPE: 4.097786
R2: 0.746844
ExplainedVar: 0.754630
MedianAE: 5655.501092
MaxError: 34905.057389
MAE_scaled: 0.344418
RMSE_scaled: 0.438179

TOTAL_OPERATING_EXP - TRAIN
MAE: 1428.953278
RMSE: 2889.792990
MAPE: 1.390182
sMAPE: 1.403129
R2: 0.855594
ExplainedVar: 0.855811
MedianAE: 987.011694
MaxError: 37645.585768
MAE_scaled: 0.187907
RMSE_scaled: 0.380008

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1673.187340
RMSE: 2966.753315
MAPE: 1.502832
sMAPE: 1.514628
R2: 0.746174
ExplainedVar: 0.752141
MedianAE: 1171.342337
MaxError: 26920.172062
MAE_scaled: 0.220024
RMSE_scaled: 0.390128

  -> no

## 18. LSTM-mTrans-MLP — training

In [20]:
if "lstm_mtrans_mlp" in MODELS_TO_RUN:
    HISTORIES["lstm_mtrans_mlp"] = train_model("lstm_mtrans_mlp")
else:
    print("lstm_mtrans_mlp skipped.")


[LSTM-mTrans-MLP] RESUME=True but no checkpoint found - starting at epoch 1.


LSTM-mTrans-MLP | Epoch 1/100

Train Loss: 0.293046
Val Loss:   0.314941
LR:         0.00100000
Epoch time: 4.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 13717.902810
RMSE: 16949.151911
MAPE: 9.856141
sMAPE: 9.633413
R2: 0.312633
ExplainedVar: 0.312682
MedianAE: 11965.688732
MaxError: 63040.819490
MAE_scaled: 0.671018
RMSE_scaled: 0.829076

NET_PATIENT_REVENUE - VALIDATION
MAE: 17823.964475
RMSE: 21051.043402
MAPE: 10.032727
sMAPE: 10.485883
R2: -0.398058
ExplainedVar: 0.044785
MedianAE: 16545.404979
MaxError: 55934.445549
MAE_scaled: 0.871868
RMSE_scaled: 1.029722

TOTAL_OPERATING_EXP - TRAIN
MAE: 4874.819163
RMSE: 6321.068930
MAPE: 4.860710
sMAPE: 4.836229
R2: 0.309073
ExplainedVar: 0.309128
MedianAE: 4155.633405
MaxError: 45339.784739
MAE_scaled: 0.641038
RMSE_scaled: 0.831220

TOTAL_OPERATING_EXP - VALIDATION
MAE: 5171.325964
RMSE: 6231.269512
MAPE: 4.643508
sMAPE: 4.708811
R2: -0.119763
ExplainedVar: 0.046555
MedianAE: 5157.281942
MaxError: 32748.982575
MAE_scaled: 0.680029
RMSE_scaled: 

LSTM-mTrans-MLP | Epoch 2/100

Train Loss: 0.180313
Val Loss:   0.154315
LR:         0.00100000
Epoch time: 4.2s

NET_PATIENT_REVENUE - TRAIN
MAE: 10624.208821
RMSE: 13253.293025
MAPE: 7.602285
sMAPE: 7.496116
R2: 0.579718
ExplainedVar: 0.580111
MedianAE: 9140.838219
MaxError: 44845.875850
MAE_scaled: 0.519688
RMSE_scaled: 0.648291

NET_PATIENT_REVENUE - VALIDATION
MAE: 12489.939526
RMSE: 15317.680038
MAPE: 7.005164
sMAPE: 7.233181
R2: 0.259775
ExplainedVar: 0.469756
MedianAE: 10913.727722
MaxError: 47655.879194
MAE_scaled: 0.610952
RMSE_scaled: 0.749272

TOTAL_OPERATING_EXP - TRAIN
MAE: 3609.507194
RMSE: 4891.262906
MAPE: 3.586811
sMAPE: 3.583179
R2: 0.586293
ExplainedVar: 0.586786
MedianAE: 2923.360404
MaxError: 40586.365358
MAE_scaled: 0.474650
RMSE_scaled: 0.643201

TOTAL_OPERATING_EXP - VALIDATION
MAE: 3257.147840
RMSE: 4298.733510
MAPE: 2.917634
sMAPE: 2.943821
R2: 0.467090
ExplainedVar: 0.514670
MedianAE: 3088.098896
MaxError: 29250.432283
MAE_scaled: 0.428315
RMSE_scaled: 0.565

LSTM-mTrans-MLP | Epoch 3/100

Train Loss: 0.105538
Val Loss:   0.073275
LR:         0.00100000
Epoch time: 4.5s

NET_PATIENT_REVENUE - TRAIN
MAE: 8035.445476
RMSE: 10186.001242
MAPE: 5.682431
sMAPE: 5.648299
R2: 0.751744
ExplainedVar: 0.751986
MedianAE: 6609.406550
MaxError: 37750.800136
MAE_scaled: 0.393058
RMSE_scaled: 0.498253

NET_PATIENT_REVENUE - VALIDATION
MAE: 8150.752651
RMSE: 10351.015084
MAPE: 4.671527
sMAPE: 4.724020
R2: 0.661979
ExplainedVar: 0.694695
MedianAE: 6763.287506
MaxError: 38069.807972
MAE_scaled: 0.398698
RMSE_scaled: 0.506325

TOTAL_OPERATING_EXP - TRAIN
MAE: 2558.806891
RMSE: 3842.917607
MAPE: 2.518005
sMAPE: 2.529352
R2: 0.744628
ExplainedVar: 0.745165
MedianAE: 1973.277565
MaxError: 40125.744456
MAE_scaled: 0.336483
RMSE_scaled: 0.505343

TOTAL_OPERATING_EXP - VALIDATION
MAE: 2181.143898
RMSE: 3350.021540
MAPE: 1.963485
sMAPE: 1.972567
R2: 0.676356
ExplainedVar: 0.682463
MedianAE: 1719.071515
MaxError: 26746.275818
MAE_scaled: 0.286820
RMSE_scaled: 0.440528

LSTM-mTrans-MLP | Epoch 4/100

Train Loss: 0.085526
Val Loss:   0.086152
LR:         0.00100000
Epoch time: 4.7s

NET_PATIENT_REVENUE - TRAIN
MAE: 7095.748033
RMSE: 9039.900468
MAPE: 5.041923
sMAPE: 5.009124
R2: 0.804467
ExplainedVar: 0.804481
MedianAE: 5782.994012
MaxError: 43229.994663
MAE_scaled: 0.347092
RMSE_scaled: 0.442191

NET_PATIENT_REVENUE - VALIDATION
MAE: 9311.301620
RMSE: 11621.666454
MAPE: 5.299444
sMAPE: 5.467127
R2: 0.573897
ExplainedVar: 0.727825
MedianAE: 8166.111406
MaxError: 40795.935099
MAE_scaled: 0.455467
RMSE_scaled: 0.568480

TOTAL_OPERATING_EXP - TRAIN
MAE: 2200.129247
RMSE: 3536.295526
MAPE: 2.165274
sMAPE: 2.175870
R2: 0.783754
ExplainedVar: 0.784095
MedianAE: 1656.067136
MaxError: 38275.075603
MAE_scaled: 0.289317
RMSE_scaled: 0.465023

TOTAL_OPERATING_EXP - VALIDATION
MAE: 2218.065406
RMSE: 3418.166540
MAPE: 1.998174
sMAPE: 2.023557
R2: 0.663055
ExplainedVar: 0.707626
MedianAE: 1715.685117
MaxError: 27708.291445
MAE_scaled: 0.291675
RMSE_scaled: 0.449489


...

LSTM-mTrans-MLP | Epoch 16/100

Train Loss: 0.065651
Val Loss:   0.075899
LR:         0.00050000
Epoch time: 4.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 6260.900506
RMSE: 8079.059996
MAPE: 4.413458
sMAPE: 4.394377
R2: 0.843824
ExplainedVar: 0.843872
MedianAE: 5234.455260
MaxError: 36278.521429
MAE_scaled: 0.306255
RMSE_scaled: 0.395191

NET_PATIENT_REVENUE - VALIDATION
MAE: 8070.194916
RMSE: 10309.631266
MAPE: 4.611980
sMAPE: 4.701811
R2: 0.664676
ExplainedVar: 0.722877
MedianAE: 6630.788187
MaxError: 40806.992003
MAE_scaled: 0.394758
RMSE_scaled: 0.504301

TOTAL_OPERATING_EXP - TRAIN
MAE: 1801.840427
RMSE: 3193.178873
MAPE: 1.763503
sMAPE: 1.775136
R2: 0.823682
ExplainedVar: 0.824035
MedianAE: 1273.960100
MaxError: 39245.022223
MAE_scaled: 0.236942
RMSE_scaled: 0.419903

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1994.752067
RMSE: 3285.901459
MAPE: 1.793878
sMAPE: 1.807770
R2: 0.688626
ExplainedVar: 0.698165
MedianAE: 1537.457418
MaxError: 28570.870917
MAE_scaled: 0.262310
RMSE_scaled: 0.432096

LSTM-mTrans-MLP | Epoch 17/100

Train Loss: 0.065556
Val Loss:   0.080364
LR:         0.00025000
Epoch time: 4.9s

NET_PATIENT_REVENUE - TRAIN
MAE: 6265.998938
RMSE: 8013.763775
MAPE: 4.435050
sMAPE: 4.412598
R2: 0.846338
ExplainedVar: 0.846338
MedianAE: 5102.792486
MaxError: 38522.127688
MAE_scaled: 0.306504
RMSE_scaled: 0.391997

NET_PATIENT_REVENUE - VALIDATION
MAE: 8471.723670
RMSE: 10822.696587
MAPE: 4.798375
sMAPE: 4.916723
R2: 0.630471
ExplainedVar: 0.725044
MedianAE: 6980.371243
MaxError: 40080.313388
MAE_scaled: 0.414399
RMSE_scaled: 0.529398

TOTAL_OPERATING_EXP - TRAIN
MAE: 1759.751590
RMSE: 3202.357297
MAPE: 1.720208
sMAPE: 1.733230
R2: 0.822667
ExplainedVar: 0.823015
MedianAE: 1255.101710
MaxError: 39875.320551
MAE_scaled: 0.231407
RMSE_scaled: 0.421110

TOTAL_OPERATING_EXP - VALIDATION
MAE: 2013.413257
RMSE: 3286.604457
MAPE: 1.805195
sMAPE: 1.822066
R2: 0.688493
ExplainedVar: 0.706621
MedianAE: 1593.633438
MaxError: 28475.710271
MAE_scaled: 0.264764
RMSE_scaled: 0.432188

LSTM-mTrans-MLP | Epoch 18/100

Train Loss: 0.063848
Val Loss:   0.077106
LR:         0.00025000
Epoch time: 4.3s

NET_PATIENT_REVENUE - TRAIN
MAE: 6188.473255
RMSE: 7971.715771
MAPE: 4.374170
sMAPE: 4.354764
R2: 0.847947
ExplainedVar: 0.848041
MedianAE: 4971.404270
MaxError: 39627.046087
MAE_scaled: 0.302712
RMSE_scaled: 0.389940

NET_PATIENT_REVENUE - VALIDATION
MAE: 8092.466649
RMSE: 10344.829800
MAPE: 4.634803
sMAPE: 4.728409
R2: 0.662383
ExplainedVar: 0.721934
MedianAE: 6740.469236
MaxError: 41894.666437
MAE_scaled: 0.395847
RMSE_scaled: 0.506022

TOTAL_OPERATING_EXP - TRAIN
MAE: 1735.366001
RMSE: 3166.629489
MAPE: 1.696780
sMAPE: 1.709054
R2: 0.826601
ExplainedVar: 0.826878
MedianAE: 1221.161477
MaxError: 39614.358893
MAE_scaled: 0.228200
RMSE_scaled: 0.416411

TOTAL_OPERATING_EXP - VALIDATION
MAE: 1988.972000
RMSE: 3298.759436
MAPE: 1.793188
sMAPE: 1.806094
R2: 0.686185
ExplainedVar: 0.692216
MedianAE: 1494.025170
MaxError: 29107.651041
MAE_scaled: 0.261550
RMSE_scaled: 0.433787

LSTM-mTrans-MLP | Epoch 19/100

Train Loss: 0.064124
Val Loss:   0.083927
LR:         0.00025000
Epoch time: 4.4s

NET_PATIENT_REVENUE - TRAIN
MAE: 6251.755280
RMSE: 7963.611161
MAPE: 4.416765
sMAPE: 4.397241
R2: 0.848256
ExplainedVar: 0.848348
MedianAE: 5194.428220
MaxError: 33361.925690
MAE_scaled: 0.305808
RMSE_scaled: 0.389544

NET_PATIENT_REVENUE - VALIDATION
MAE: 8447.648434
RMSE: 10792.254933
MAPE: 4.801310
sMAPE: 4.919359
R2: 0.632547
ExplainedVar: 0.722944
MedianAE: 7156.232702
MaxError: 41868.983380
MAE_scaled: 0.413221
RMSE_scaled: 0.527908

TOTAL_OPERATING_EXP - TRAIN
MAE: 1734.127470
RMSE: 3159.272218
MAPE: 1.697287
sMAPE: 1.708708
R2: 0.827406
ExplainedVar: 0.827535
MedianAE: 1207.915244
MaxError: 39090.051559
MAE_scaled: 0.228038
RMSE_scaled: 0.415444

TOTAL_OPERATING_EXP - VALIDATION
MAE: 2117.955466
RMSE: 3387.089224
MAPE: 1.897646
sMAPE: 1.918094
R2: 0.669154
ExplainedVar: 0.696170
MedianAE: 1691.481051
MaxError: 29326.492317
MAE_scaled: 0.278511
RMSE_scaled: 0.445402

LSTM-mTrans-MLP | Epoch 20/100

Train Loss: 0.065371
Val Loss:   0.082211
LR:         0.00025000
Epoch time: 4.5s

NET_PATIENT_REVENUE - TRAIN
MAE: 6270.134888
RMSE: 8059.163432
MAPE: 4.447390
sMAPE: 4.424623
R2: 0.844592
ExplainedVar: 0.844610
MedianAE: 5057.891119
MaxError: 38769.877588
MAE_scaled: 0.306707
RMSE_scaled: 0.394218

NET_PATIENT_REVENUE - VALIDATION
MAE: 8397.801347
RMSE: 10720.677344
MAPE: 4.781376
sMAPE: 4.900990
R2: 0.637405
ExplainedVar: 0.728673
MedianAE: 7126.445662
MaxError: 42289.881191
MAE_scaled: 0.410783
RMSE_scaled: 0.524407

TOTAL_OPERATING_EXP - TRAIN
MAE: 1754.490678
RMSE: 3185.319828
MAPE: 1.715958
sMAPE: 1.728459
R2: 0.824549
ExplainedVar: 0.824862
MedianAE: 1225.614822
MaxError: 40233.899829
MAE_scaled: 0.230715
RMSE_scaled: 0.418869

TOTAL_OPERATING_EXP - VALIDATION
MAE: 2049.253927
RMSE: 3345.002354
MAPE: 1.840442
sMAPE: 1.860164
R2: 0.677325
ExplainedVar: 0.700189
MedianAE: 1572.463858
MaxError: 29490.791687
MAE_scaled: 0.269477
RMSE_scaled: 0.439868

## 19. Final test evaluation — BEST checkpoint, evaluated once

The best (lowest-validation-loss) checkpoint is loaded for each model and the
test set is scored **once**. The latest checkpoint is never substituted.
Predictions are written in chronological test order, with dates.

In [21]:
TEST_RESULTS, TEST_PRED = {}, {}

for mk in MODELS_TO_RUN:
    name = MODEL_DISPLAY[mk]
    model, bep, bvl = load_best_for_eval(mk)
    print(f"[{name}] loaded BEST checkpoint from epoch {bep} (val loss {bvl:.6f})")

    tloss, yts, yps = evaluate(model, test_loader)
    yto, ypo = to_original_units(yts), to_original_units(yps)
    per_t = metrics_all(yts, yps)
    TEST_RESULTS[mk] = per_t
    TEST_PRED[mk] = (yto, ypo)

    print("\n" + "=" * 60)
    print(f"FINAL TEST RESULTS - {name}")
    print("=" * 60)
    print(f"(test loss in scaled target space: {tloss:.6f})")
    for t in TARGETS:
        print(f"\n{t}")
        for k in METRIC_NAMES:
            print(f"{k}: {per_t[t][k]:.6f}")
    print()

    with open(mdir(mk, "metrics", "test_metrics.json"), "w") as f:
        json.dump({"best_epoch": bep, "best_val_loss": bvl,
                   "test_loss_scaled_space": tloss, "metrics": per_t}, f, indent=2)

    pdf = pd.DataFrame({"index": np.arange(len(yto)),
                        "DATE": pd.to_datetime(dt_test)})
    for i, t in enumerate(TARGETS):
        pdf[f"actual_{t}"] = yto[:, i]
        pdf[f"predicted_{t}"] = ypo[:, i]
    pdf.to_csv(mdir(mk, "predictions", "test_predictions.csv"), index=False)
    print("Saved:", mdir(mk, "predictions", "test_predictions.csv"))

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


[CNN-LSTM] loaded BEST checkpoint from epoch 16 (val loss 0.053460)

FINAL TEST RESULTS - CNN-LSTM
(test loss in scaled target space: 0.079470)

NET_PATIENT_REVENUE
MAE: 8008.692913
RMSE: 10444.682150
MAPE: 4.338768
sMAPE: 4.378503
R2: 0.635655
ExplainedVar: 0.671673
MedianAE: 6424.259801
MaxError: 40182.709636
MAE_scaled: 0.391749
RMSE_scaled: 0.510907

TOTAL_OPERATING_EXP
MAE: 2278.787943
RMSE: 4305.721145
MAPE: 1.951753
sMAPE: 1.980868
R2: 0.504509
ExplainedVar: 0.529112
MedianAE: 1621.358820
MaxError: 47451.472794
MAE_scaled: 0.299660
RMSE_scaled: 0.566202

Saved: benchmarks_repo_data/baseline_outputs_v1/cnn_lstm/predictions/test_predictions.csv
[TCN-LSTM] loaded BEST checkpoint from epoch 29 (val loss 0.052840)



FINAL TEST RESULTS - TCN-LSTM
(test loss in scaled target space: 0.079455)

NET_PATIENT_REVENUE
MAE: 8026.693478
RMSE: 10490.923084
MAPE: 4.329653
sMAPE: 4.371094
R2: 0.632422
ExplainedVar: 0.672070
MedianAE: 6630.276875
MaxError: 42930.341585
MAE_scaled: 0.392630
RMSE_scaled: 0.513169

TOTAL_OPERATING_EXP
MAE: 2374.378290
RMSE: 4359.327507
MAPE: 2.026158
sMAPE: 2.061009
R2: 0.492094
ExplainedVar: 0.534292
MedianAE: 1668.142730
MaxError: 47579.043933
MAE_scaled: 0.312231
RMSE_scaled: 0.573251

Saved: benchmarks_repo_data/baseline_outputs_v1/tcn_lstm/predictions/test_predictions.csv
[LSTM-mTrans-MLP] loaded BEST checkpoint from epoch 5 (val loss 0.062664)


/usr/local/lib64/python3.9/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



FINAL TEST RESULTS - LSTM-mTrans-MLP
(test loss in scaled target space: 0.091300)

NET_PATIENT_REVENUE
MAE: 9717.799891
RMSE: 12231.683511
MAPE: 5.203485
sMAPE: 5.314807
R2: 0.500317
ExplainedVar: 0.621082
MedianAE: 8184.299950
MaxError: 40002.354574
MAE_scaled: 0.475351
RMSE_scaled: 0.598319

TOTAL_OPERATING_EXP
MAE: 2336.324520
RMSE: 4352.053556
MAPE: 2.013704
sMAPE: 2.035830
R2: 0.493788
ExplainedVar: 0.501040
MedianAE: 1678.547970
MaxError: 46909.494621
MAE_scaled: 0.307226
RMSE_scaled: 0.572295

Saved: benchmarks_repo_data/baseline_outputs_v1/lstm_mtrans_mlp/predictions/test_predictions.csv


## 20. Test metrics CSV and model comparison

In [22]:
rows = []
for mk in MODELS_TO_RUN:
    for t in TARGETS:
        r = {"model": MODEL_DISPLAY[mk], "target": t}
        r.update({k: TEST_RESULTS[mk][t][k] for k in METRIC_NAMES})
        rows.append(r)

test_metrics_df = pd.DataFrame(rows, columns=["model", "target"] + METRIC_NAMES)
test_metrics_df.to_csv(OUT_P / "comparison" / "test_metrics.csv", index=False)
test_metrics_df.to_csv(OUT_P / "comparison" / "model_comparison.csv", index=False)

print("Saved:", OUT_P / "comparison" / "test_metrics.csv")
print("Saved:", OUT_P / "comparison" / "model_comparison.csv")
print("\nTarget-wise results kept separate (never averaged into one score).\n")
with pd.option_context("display.max_columns", None, "display.width", 250):
    print(test_metrics_df.to_string(index=False))


Saved: benchmarks_repo_data/baseline_outputs_v1/comparison/test_metrics.csv
Saved: benchmarks_repo_data/baseline_outputs_v1/comparison/model_comparison.csv

Target-wise results kept separate (never averaged into one score).

          model              target         MAE         RMSE     MAPE    sMAPE       R2  ExplainedVar    MedianAE     MaxError  MAE_scaled  RMSE_scaled
       CNN-LSTM NET_PATIENT_REVENUE 8008.692913 10444.682150 4.338768 4.378503 0.635655      0.671673 6424.259801 40182.709636    0.391749     0.510907
       CNN-LSTM TOTAL_OPERATING_EXP 2278.787943  4305.721145 1.951753 1.980868 0.504509      0.529112 1621.358820 47451.472794    0.299660     0.566202
       TCN-LSTM NET_PATIENT_REVENUE 8026.693478 10490.923084 4.329653 4.371094 0.632422      0.672070 6630.276875 42930.341585    0.392630     0.513169
       TCN-LSTM TOTAL_OPERATING_EXP 2374.378290  4359.327507 2.026158 2.061009 0.492094      0.534292 1668.142730 47579.043933    0.312231     0.573251
LSTM-mTrans-MLP

## 21. Plots — training curves, actual vs predicted, residuals, comparison

All figures: font size 20, DPI 300, saved to each model's `plots/` folder
(rendered to file rather than inline to keep the notebook small).

In [23]:
def _save(fig, path):
    fig.tight_layout(); fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)


def training_curves(mk):
    hp = mdir(mk, "metrics", "history.csv")
    if not hp.exists():
        print(f"  no history for {MODEL_DISPLAY[mk]}"); return
    h = pd.read_csv(hp); name = MODEL_DISPLAY[mk]; od = mdir(mk, "plots")

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(h.epoch, h.train_loss, lw=2.5, label="Train")
    ax.plot(h.epoch, h.val_loss, lw=2.5, label="Validation")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (SmoothL1)")
    ax.set_title(f"{name} - Loss"); ax.legend(); ax.grid(alpha=.3)
    _save(fig, od / "loss_curve.png")

    for metric in ("R2", "MAE", "RMSE"):
        for t in TARGETS:
            tc, vc = f"train_{t}_{metric}", f"val_{t}_{metric}"
            if tc not in h.columns:
                continue
            fig, ax = plt.subplots(figsize=(12, 7))
            ax.plot(h.epoch, h[tc], lw=2.5, label="Train")
            ax.plot(h.epoch, h[vc], lw=2.5, label="Validation")
            ax.set_xlabel("Epoch"); ax.set_ylabel(metric)
            ax.set_title(f"{name}\n{t} - {metric}", fontsize=20)
            ax.legend(); ax.grid(alpha=.3)
            _save(fig, od / f"{metric.lower()}_curve_{t}.png")

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(h.epoch, h.learning_rate, lw=2.5, color="green")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Learning rate"); ax.set_yscale("log")
    ax.set_title(f"{name} - Learning Rate"); ax.grid(alpha=.3)
    _save(fig, od / "learning_rate.png")
    print(f"  {name}: training curves saved")


def actual_vs_pred(mk):
    yto, ypo = TEST_PRED[mk]; name = MODEL_DISPLAY[mk]; od = mdir(mk, "plots")
    dts = pd.to_datetime(dt_test)
    for i, t in enumerate(TARGETS):
        fig, ax = plt.subplots(figsize=(16, 7))
        ax.plot(dts, yto[:, i], lw=2.0, label="Actual")
        ax.plot(dts, ypo[:, i], lw=2.0, label="Predicted", alpha=.85)
        ax.set_xlabel("Test date (chronological)"); ax.set_ylabel(t)
        ax.set_title(f"{name}\n{t} - Actual vs Predicted", fontsize=20)
        ax.legend(); ax.grid(alpha=.3)
        fig.autofmt_xdate()
        _save(fig, od / f"test_actual_vs_predicted_{t}.png")
    print(f"  {name}: actual-vs-predicted saved")


def residuals(mk):
    yto, ypo = TEST_PRED[mk]; name = MODEL_DISPLAY[mk]; od = mdir(mk, "plots")
    res = yto - ypo; dts = pd.to_datetime(dt_test)
    for i, t in enumerate(TARGETS):
        r = res[:, i]
        fig, ax = plt.subplots(figsize=(16, 7))
        ax.plot(dts, r, lw=1.8); ax.axhline(0, ls="--", lw=2, color="black")
        ax.set_xlabel("Test date (chronological)")
        ax.set_ylabel("Residual (actual - predicted)")
        ax.set_title(f"{name}\n{t} - Residual over test period", fontsize=20)
        ax.grid(alpha=.3); fig.autofmt_xdate()
        _save(fig, od / f"residual_over_time_{t}.png")

        fig, ax = plt.subplots(figsize=(12, 7))
        ax.hist(r, bins=40, edgecolor="black")
        ax.axvline(0, ls="--", lw=2, color="black")
        ax.set_xlabel("Residual (actual - predicted)"); ax.set_ylabel("Frequency")
        ax.set_title(f"{name}\n{t} - Residual distribution", fontsize=20)
        ax.grid(alpha=.3)
        _save(fig, od / f"residual_distribution_{t}.png")
    print(f"  {name}: residual plots saved")


print("Training curves:")
for mk in MODELS_TO_RUN: training_curves(mk)
print("Actual vs predicted:")
for mk in MODELS_TO_RUN: actual_vs_pred(mk)
print("Residual analysis (descriptive diagnostics only - no distributional claim):")
for mk in MODELS_TO_RUN: residuals(mk)


Training curves:


  CNN-LSTM: training curves saved


  TCN-LSTM: training curves saved


  LSTM-mTrans-MLP: training curves saved
Actual vs predicted:


  CNN-LSTM: actual-vs-predicted saved


  TCN-LSTM: actual-vs-predicted saved


  LSTM-mTrans-MLP: actual-vs-predicted saved
Residual analysis (descriptive diagnostics only - no distributional claim):


  CNN-LSTM: residual plots saved


  TCN-LSTM: residual plots saved


  LSTM-mTrans-MLP: residual plots saved


In [24]:
cmp_dir = OUT_P / "comparison" / "plots"

def comparison_plot(metric):
    for t in TARGETS:
        names = [MODEL_DISPLAY[mk] for mk in MODELS_TO_RUN]
        vals = [TEST_RESULTS[mk][t][metric] for mk in MODELS_TO_RUN]
        fig, ax = plt.subplots(figsize=(12, 7))
        bars = ax.bar(names, vals, edgecolor="black")
        ax.set_ylabel(metric); ax.set_title(f"{t}\nTest {metric} by model", fontsize=20)
        ax.grid(alpha=.3, axis="y"); ax.tick_params(axis="x", rotation=15)
        fin = [v for v in vals if np.isfinite(v)]
        if fin:
            span = (max(fin) - min(min(fin), 0)) or 1.0
            for b, v in zip(bars, vals):
                if np.isfinite(v):
                    ax.text(b.get_x() + b.get_width()/2, v + .02*span,
                            f"{v:.4g}", ha="center", va="bottom", fontsize=16)
        _save(fig, cmp_dir / f"comparison_{metric}_{t}.png")

for m in ("R2", "MAE", "RMSE"):
    comparison_plot(m); print(f"  {m} comparison saved")
print("Saved to:", cmp_dir)


  R2 comparison saved


  MAE comparison saved


  RMSE comparison saved
Saved to: benchmarks_repo_data/baseline_outputs_v1/comparison/plots


## 22. Final summary table

Every value below comes from actual test predictions produced by each model's
best checkpoint on the untouched test period (2024-07-01 … 2025-12-31).

In [25]:
print("=" * 72)
print("FINAL TEST SUMMARY - all models, both targets")
print("=" * 72 + "\n")
with pd.option_context("display.max_columns", None, "display.width", 300,
                       "display.float_format", lambda v: f"{v:,.4f}"):
    print(test_metrics_df.to_string(index=False))

print("\nInterpretation notes:")
print("  - 8 primary metrics are in ORIGINAL dollar units.")
print("  - MAE_scaled / RMSE_scaled divide by the TRAIN target std:")
for i, t in enumerate(TARGETS):
    print(f"      {t}: train_std = {TRAIN_TARGET_STD[i]:,.4f}")
print(f"  - MAPE denominator floor = {MAPE_EPSILON}; both targets are >> 0 here, "
      "so MAPE is well behaved.")
print(f"  - Test period: {pd.to_datetime(dt_test).min().date()} .. "
      f"{pd.to_datetime(dt_test).max().date()} ({len(dt_test)} days)")

print("\nBest-epoch summary:")
for mk in MODELS_TO_RUN:
    info = json.loads((mdir(mk, "checkpoints", "best", "best_model_info.json")).read_text())
    h = pd.read_csv(mdir(mk, "metrics", "history.csv"))
    print(f"  {MODEL_DISPLAY[mk]:18s} best_epoch={info['best_epoch']:3d} "
          f"best_val_loss={info['best_validation_loss']:.6f} "
          f"epochs_run={len(h)}")


FINAL TEST SUMMARY - all models, both targets

          model              target        MAE        RMSE   MAPE  sMAPE     R2  ExplainedVar   MedianAE    MaxError  MAE_scaled  RMSE_scaled
       CNN-LSTM NET_PATIENT_REVENUE 8,008.6929 10,444.6821 4.3388 4.3785 0.6357        0.6717 6,424.2598 40,182.7096      0.3917       0.5109
       CNN-LSTM TOTAL_OPERATING_EXP 2,278.7879  4,305.7211 1.9518 1.9809 0.5045        0.5291 1,621.3588 47,451.4728      0.2997       0.5662
       TCN-LSTM NET_PATIENT_REVENUE 8,026.6935 10,490.9231 4.3297 4.3711 0.6324        0.6721 6,630.2769 42,930.3416      0.3926       0.5132
       TCN-LSTM TOTAL_OPERATING_EXP 2,374.3783  4,359.3275 2.0262 2.0610 0.4921        0.5343 1,668.1427 47,579.0439      0.3122       0.5733
LSTM-mTrans-MLP NET_PATIENT_REVENUE 9,717.7999 12,231.6835 5.2035 5.3148 0.5003        0.6211 8,184.3000 40,002.3546      0.4754       0.5983
LSTM-mTrans-MLP TOTAL_OPERATING_EXP 2,336.3245  4,352.0536 2.0137 2.0358 0.4938        0.5010 1,678.5

## 23. Output manifest

In [26]:
EXPECTED = {
    "latest checkpoint": ("checkpoints", "latest", "latest.pt"),
    "best checkpoint": ("checkpoints", "best", "best.pt"),
    "best_model_info.json": ("checkpoints", "best", "best_model_info.json"),
    "history.csv": ("metrics", "history.csv"),
    "test_metrics.json": ("metrics", "test_metrics.json"),
    "test_predictions.csv": ("predictions", "test_predictions.csv"),
    "config.json": ("config", "config.json"),
    "model_summary.json": ("config", "model_summary.json"),
}

print("=" * 60)
print("EXPERIMENT OUTPUT MANIFEST")
print("=" * 60)
for mk in MODEL_KEYS:
    print(f"\n{MODEL_DISPLAY[mk]}")
    if mk not in MODELS_TO_RUN:
        print("  (not run)"); continue
    for lbl, parts in EXPECTED.items():
        print(f"  [{'OK' if mdir(mk, *parts).exists() else 'MISSING'}] {lbl}")
    pngs = sorted(mdir(mk, "plots").glob("*.png"))
    print(f"  [{'OK' if pngs else 'MISSING'}] plots ({len(pngs)} png)")
    for p in pngs:
        print(f"        {p.name}")

print("\nCOMPARISON")
for lbl, p in [("test_metrics.csv", OUT_P/"comparison"/"test_metrics.csv"),
               ("model_comparison.csv", OUT_P/"comparison"/"model_comparison.csv"),
               ("model_summary.json", OUT_P/"comparison"/"model_summary.json")]:
    print(f"  [{'OK' if p.exists() else 'MISSING'}] {lbl}")
cp = sorted((OUT_P/"comparison"/"plots").glob("*.png"))
print(f"  [{'OK' if cp else 'MISSING'}] comparison plots ({len(cp)} png)")
for p in cp:
    print(f"        {p.name}")
print("\nAll outputs under:", OUT_P.resolve())


EXPERIMENT OUTPUT MANIFEST

CNN-LSTM
  [OK] latest checkpoint
  [OK] best checkpoint
  [OK] best_model_info.json
  [OK] history.csv
  [OK] test_metrics.json
  [OK] test_predictions.csv
  [OK] config.json
  [OK] model_summary.json
  [OK] plots (14 png)
        learning_rate.png
        loss_curve.png
        mae_curve_NET_PATIENT_REVENUE.png
        mae_curve_TOTAL_OPERATING_EXP.png
        r2_curve_NET_PATIENT_REVENUE.png
        r2_curve_TOTAL_OPERATING_EXP.png
        residual_distribution_NET_PATIENT_REVENUE.png
        residual_distribution_TOTAL_OPERATING_EXP.png
        residual_over_time_NET_PATIENT_REVENUE.png
        residual_over_time_TOTAL_OPERATING_EXP.png
        rmse_curve_NET_PATIENT_REVENUE.png
        rmse_curve_TOTAL_OPERATING_EXP.png
        test_actual_vs_predicted_NET_PATIENT_REVENUE.png
        test_actual_vs_predicted_TOTAL_OPERATING_EXP.png

TCN-LSTM
  [OK] latest checkpoint
  [OK] best checkpoint
  [OK] best_model_info.json
  [OK] history.csv
  [OK] test_metric

## 24. Interpretation boundaries

Three categories that must not be conflated in the write-up:

**1. Paper facts.** Verified: both works exist, their venues, and their
high-level composition (CNN/TCN+LSTM hybrids compared in the first; LSTM +
modified Transformer + MLP in the second). Their *reported* results belong to
*their* datasets (traffic, air quality, financial series) and are not comparable
to results here.

**2. Implementation decisions.** Every layer size, channel/kernel/dilation
choice, head count, dropout rate, the shared `SmoothL1Loss`, `AdamW`,
`ReduceLROnPlateau` settings, gradient clipping, and the pre-norm reconstruction
of the "modified Transformer". None is claimed to be paper-specified.

**3. Experimental results.** Whatever the tables and plots above show. There is
no target R² and no threshold any model is expected to clear. Regression
forecasting has no generic "accuracy" metric, so none is reported.

A benchmark beating the proposed model is as legitimate a result as the reverse,
and should be reported as such.

### Known caveat worth stating in the paper

Both targets shift upward across the splits (train mean revenue ≈ \$143k → val
≈ \$173k → test ≈ \$183k; expenses ≈ \$101k → \$110k → \$114k). The target
transform is fitted on train only — correct for leakage avoidance, but it means
every model extrapolates into a higher regime on test. This affects all models
equally and should be disclosed rather than presented as model weakness.
